<a href="https://colab.research.google.com/github/TheGenesisAIStory/ml-trading-thesis-bot/blob/main/Company_Valuatio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. ⚙️ Setup & Config

In questa sezione configuriamo l'ambiente sperimentale per il motore DCF a 10 anni, definendo parametri, universo, logging e stile grafico coerente con una dashboard fintech moderna.

| Cell | What it does | Output |
|------|---------------|--------|
| 0.1  | Installa le dipendenze necessarie in modo silenzioso | Librerie pronte all'uso |
| 0.2  | Importa moduli, imposta il seed e lo stile grafico | Ambiente coerente e replicabile |
| 0.3  | Definisce EXPERIMENT, UNIVERSE e colori | Configurazione centrale dell'esperimento |
| 0.4  | Crea le cartelle di output e inizializza il logger | Struttura `output/` e log dell'esecuzione |

Il cuore economico di questo notebook è un modello di Discounted Cash Flow (DCF) multi‑periodo. Valutiamo ogni titolo come il valore attuale dei flussi di cassa futuri, scontati a un tasso coerente con il rischio:

\[
V_0 = \sum_{t=1}^{T} \frac{FCF_t}{(1 + r)^t} + \frac{TV}{(1 + r)^T}
\]

dove \(FCF_t\) è il free cash flow al tempo \(t\), \(r\) è il tasso di sconto (cost of equity o WACC) e \(TV\) è il terminal value, calcolato con modelli di crescita perpetua o multipli di mercato.

In [100]:
# 0.1 Installazione dipendenze
import subprocess
import sys

def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    except Exception as e:
        print(f"[WARNING] Failed to install {pkg}: {e}")

required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "python-dotenv",
    "tqdm",
    "scikit-learn",
]

for pkg in required_packages:
    pip_install(pkg)

In [101]:
# 0.2 Import, seed, stile grafici
import os
from pathlib import Path
import logging
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()
np.random.seed(42)

COLORS = {
    "primary":  "#01696f",
    "accent":   "#da7101",
    "q1":       "#c0392b",
    "neutral":  "#7a7974",
    "bg":       "#f7f6f2",
    "blue":     "#006494",
    "gold":     "#d19900",
    "purple":   "#7a39bb",
}

plt.rcParams["figure.dpi"] = 180
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["axes.facecolor"] = COLORS["bg"]
plt.rcParams["savefig.facecolor"] = COLORS["bg"]

In [102]:
# ============================================================
# 0.1 MASTER USER CONFIG
# notebook valuation / ML - enterprise grade
# ============================================================

USER_SELECTION = {
    # --------------------------------------------------------
    # A) DATA SOURCE / COMPANY
    # --------------------------------------------------------
    "data_source_choice": "europe_italy_market_data",
    # options:
    # "default"
    # "database_finanziario"
    # "ml_trading_repo"
    # "europe_stoxx_companies"
    # "europe_italy_market_data"
    # "sqlite_database"
    # "custom_folder"

    "selected_company": "ucg_mi",
    "custom_folder_path": None,

    # --------------------------------------------------------
    # B) USER EXPERIENCE
    # --------------------------------------------------------
    "user_profile": "cfa_balanced",
    # options:
    # "retail_simple"
    # "conservative"
    # "balanced"
    # "aggressive"
    # "cfa_balanced"
    # "cfa_advanced"

    # --------------------------------------------------------
    # C) MODEL SCOPE
    # --------------------------------------------------------
    "run_valuation_models": [
        "dcf",
        "relative_valuation",
        "residual_income",
        "dividend_discount_model",
        "economic_profit"
    ],
    # possible:
    # "dcf"
    # "relative_valuation"
    # "residual_income"
    # "dividend_discount_model"
    # "economic_profit"

    "run_ml_models": [
        "linear_regression",
        "ridge",
        "lasso",
        "elastic_net",
        "random_forest",
        "xgboost",
        "lightgbm",
        "catboost"
    ],
    # possible:
    # "linear_regression"
    # "ridge"
    # "lasso"
    # "elastic_net"
    # "random_forest"
    # "xgboost"
    # "lightgbm"
    # "catboost"

    "compare_models": True,
    "ensemble_models": False,

    # --------------------------------------------------------
    # D) TARGET DEFINITION
    # --------------------------------------------------------
    "valuation_target": "fair_value",
    # options:
    # "fair_value"
    # "expected_return_12m"
    # "upside_downside_pct"
    # "valuation_gap"

    "ml_task_type": "regression",
    # options:
    # "regression"
    # "classification"

    "model_selection_metric": "out_of_sample_rmse",
    "ranking_metric": "blended_score",

    # --------------------------------------------------------
    # E) PEERS
    # --------------------------------------------------------
    "peer_selection_method": "sector_country",
    # options:
    # "sector_only"
    # "sector_country"
    # "sector_size_country"
    # "manual"

    "n_peers": 7,
    "manual_peers": [],

    # --------------------------------------------------------
    # F) REPORTING
    # --------------------------------------------------------
    "save_intermediate_tables": True,
    "save_figures": True,
    "save_model_objects": False,
    "verbose_logging": True,
}

In [103]:
# ============================================================
# 0.2 PRESET MAP
# ============================================================

PRESET_MAP = {
    "retail_simple": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.35,
        "sales_growth_rate": 0.04,
        "target_operating_margin": 0.15,
        "ml_primary_model": "random_forest",
        "n_peers": 5,
        "use_sector_default_assumptions": True,
    },
    "conservative": {
        "discount_rate": 0.10,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.055,
        "terminal_growth_rate": 0.015,
        "forecast_horizon_years": 5,
        "tax_rate": 0.30,
        "reinvestment_rate": 0.45,
        "sales_growth_rate": 0.03,
        "target_operating_margin": 0.13,
        "ml_primary_model": "elastic_net",
        "n_peers": 6,
        "use_sector_default_assumptions": True,
    },
    "balanced": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.40,
        "sales_growth_rate": 0.05,
        "target_operating_margin": 0.16,
        "ml_primary_model": "random_forest",
        "n_peers": 5,
        "use_sector_default_assumptions": True,
    },
    "aggressive": {
        "discount_rate": 0.08,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.045,
        "terminal_growth_rate": 0.025,
        "forecast_horizon_years": 7,
        "tax_rate": 0.27,
        "reinvestment_rate": 0.50,
        "sales_growth_rate": 0.07,
        "target_operating_margin": 0.18,
        "ml_primary_model": "xgboost",
        "n_peers": 8,
        "use_sector_default_assumptions": True,
    },
    "cfa_balanced": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.42,
        "sales_growth_rate": 0.05,
        "target_operating_margin": 0.17,
        "return_on_new_invested_capital": 0.11,
        "target_debt_to_capital": 0.30,
        "cost_of_debt": 0.045,
        "beta": 1.00,
        "ml_primary_model": "random_forest",
        "n_peers": 7,
        "use_sector_default_assumptions": True,
    },
    "cfa_advanced": {
        "discount_rate": 0.095,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.055,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 7,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.45,
        "sales_growth_rate": 0.055,
        "target_operating_margin": 0.18,
        "return_on_new_invested_capital": 0.12,
        "target_debt_to_capital": 0.32,
        "cost_of_debt": 0.0475,
        "beta": 1.05,
        "ml_primary_model": "xgboost",
        "n_peers": 10,
        "use_sector_default_assumptions": True,
    },
}

In [104]:
# ============================================================
# 0.3 ADVANCED VALUATION / ML CONFIG
# ============================================================

VALUATION_CONFIG = {
    "dcf": {
        "cash_flow_definition": "fcff",
        # "fcff" / "fcfe"
        "discounting_approach": "wacc",
        # "wacc" / "cost_of_equity"
        "terminal_value_method": "gordon_growth",
        # "gordon_growth" / "exit_multiple"
        "use_midyear_convention": True,
        "normalize_margins": True,
        "normalize_working_capital": True,
        "normalize_capex": True,
    },
    "relative_valuation": {
        "multiple_set": ["pe", "ev_ebitda", "ev_sales", "pbv"],
        "aggregation_method": "median",
        # "mean" / "median" / "trimmed_mean"
        "outlier_filtering": True,
        "winsorize_percentile": 0.05,
    },
    "residual_income": {
        "book_value_anchor": True,
        "cost_of_equity_method": "capm",
    },
    "dividend_discount_model": {
        "enabled_only_for_dividend_payers": True,
        "dividend_growth_method": "historical_blended",
    },
    "economic_profit": {
        "capital_charge_method": "wacc_times_invested_capital",
    },
    "sum_of_the_parts": {
        "enabled_for_conglomerates_only": True,
        "segment_valuation_method": "blended",
    },
    "asset_based_valuation": {
        "use_tangible_book_focus": True,
        "apply_holding_discount": False,
    },
}

In [105]:
ML_CONFIG = {
    "feature_set_type": "fundamental_plus_market",
    # "fundamental_only"
    # "market_only"
    # "fundamental_plus_market"
    # "fundamental_market_macro"

    "feature_selection_method": "hybrid",
    # "none"
    # "correlation_filter"
    # "mutual_info"
    # "model_based"
    # "hybrid"

    "scaling_method": "robust",
    # "none"
    # "standard"
    # "minmax"
    # "robust"

    "train_window_months": 60,
    "validation_window_months": 12,
    "test_window_months": 12,
    "cross_validation_folds": 5,
    "time_series_split": True,
    "hyperparameter_tuning": True,
    "performance_metric_regression": "rmse",
    # "rmse" / "mae" / "mape" / "r2"

    "performance_metric_classification": "f1",
    # "accuracy" / "precision" / "recall" / "f1" / "roc_auc"

    "classification_labels": ["SELL", "HOLD", "BUY"],
    "prediction_horizon_months": 12,
}

In [128]:
# 0.3 Configurazione PROJECT_ROOT, EXPERIMENT, UNIVERSE

# Se lavori in Colab con Drive montato:
# PROJECT_ROOT = Path("/content/drive/MyDrive/machine-learning-for-trading").resolve()
# Se lavori in locale, puoi usare Path(".").resolve() dalla root del repo:
PROJECT_ROOT = Path(".").resolve()

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

EXPERIMENT = {
    "name": "fundamental_valuation_dcf_definitive",
    "start_date": "2010-01-01",
    "end_date":   "2026-01-01",
    "target":     "future_excess_return_12m",
    "horizons":   [252],
    "test_start": "2018-01-01",
    "embargo":    21,
    "n_quantiles": 5,
    "cost_bps":   10.0,
    "models":     ["ols"],
    "run_ablation":  True,
    "run_backtest":  True,
    "save_figures":  True,
    "feature_blocks": [
        "dcf_inputs",
        "quality_ratios",
        "valuation_outputs"
    ],
    # selezione sorgente per il file pulito
    # "file_mode": "manual" oppure "manifest" se vuoi usare un manifest in futuro
    "file_mode": "manual",
    # placeholder: questi li imposterai tu in Section 1
    "clean_panel_path": None,
    # DCF specific
    "dcf_horizon_years": 10,
    "dcf_terminal_method": ["gordon_growth", "exit_multiple"],
    "dcf_perpetual_growth": 0.02,
    "dcf_exit_multiple_type": "ev_ebitda",
    "dcf_exit_multiple_default": 12.0,
    # Discount rate parameters
    "discount_rate_mode": "ke_capm",
    "risk_free_proxy": "US10Y",
    "equity_risk_premium": 0.05,
    "beta_lookback_years": 5,
    "active_universe": "best_universe_118",
    "price_end_date": "2026-05-13",
    "max_tickers_api": 200,
    "tax_rate": 0.25
}

UNIVERSES = {
    "best_universe_118": {
        "description": "118 titoli Best Universe (Finance Database)",
        "fund_file": "TimeSeriesCleanStocksItems.xlsx",
        "update_files": [
            "Aggiornamento database foundamentals Best Universe 2+.xlsx",
            "Aggiornamento database foundamentals Best Universe.xlsx"
        ],
        "price_source": "drive+yfinance"
    },
    "ftse_mib": {
        "description": "FTSE MIB 40 costituenti",
        "price_source": "yfinance",
        "suffix": ".MI"
    },
    "sp500": {
        "description": "S&P 500",
        "price_source": "yfinance",
        "suffix": ""
    },
    "eurostoxx50": {
        "description": "EuroStoxx 50",
        "fund_file": "EUROSTOXX50.xlsx",
        "price_source": "yfinance",
        "suffix": ""
    }
}

PROJECT_ROOT = /content


In [106]:
GOVERNANCE_CONFIG = {
    "random_seed": 42,
    "run_sensitivity_analysis": True,
    "sensitivity_discount_rate_bps": [-200, -100, 0, 100, 200],
    "sensitivity_terminal_growth_bps": [-100, 0, 100],
    "run_scenario_analysis": True,
    "scenario_names": ["bear", "base", "bull"],
    "generate_investment_view": True,
    "generate_model_comparison_table": True,
    "generate_peer_comparison_table": True,
    "generate_explainability_outputs": True,
    "compare_train_vs_test_metrics": True,
    "require_out_of_sample_validation": True,
    "store_model_ranking": True,
}

In [108]:
# ============================================================
# 0.6 RE-MERGE CONFIG AFTER MANUAL UPDATES
# ============================================================

selected_profile = USER_SELECTION["user_profile"]

if selected_profile not in PRESET_MAP:
    raise ValueError(
        f"user_profile '{selected_profile}' non valido. "
        f"Valori ammessi: {list(PRESET_MAP.keys())}"
    )

preset_values = PRESET_MAP[selected_profile]

EXPERIMENT.update(USER_SELECTION)
EXPERIMENT.update(preset_values)
EXPERIMENT["VALUATION_CONFIG"] = VALUATION_CONFIG
EXPERIMENT["ML_CONFIG"] = ML_CONFIG
EXPERIMENT["GOVERNANCE_CONFIG"] = GOVERNANCE_CONFIG

print("Configurazione aggiornata correttamente.")
print(f"Selected company: {EXPERIMENT['selected_company']}")
print(f"User profile: {EXPERIMENT['user_profile']}")
print(f"Valuation models: {EXPERIMENT['run_valuation_models']}")
print(f"ML models: {EXPERIMENT['run_ml_models']}")
print(f"Model selection metric: {EXPERIMENT.get('model_selection_metric', 'N/A')}")
print(f"Ranking metric: {EXPERIMENT.get('ranking_metric', 'N/A')}")

Configurazione aggiornata correttamente.
Selected company: ucg_mi
User profile: cfa_balanced
Valuation models: ['dcf', 'relative_valuation', 'residual_income', 'dividend_discount_model', 'economic_profit']
ML models: ['linear_regression', 'ridge', 'lasso', 'elastic_net', 'random_forest', 'xgboost', 'lightgbm', 'catboost']
Model selection metric: out_of_sample_rmse
Ranking metric: blended_score


In [ ]:
# ============================================================
# 0.6 bis - Reframe del notebook: company analysis + ML overlay
# ============================================================

EXPERIMENT["analysis_mode"] = "company_analysis_with_ml_overlay"
EXPERIMENT["primary_objective"] = "worth_investing_decision"
EXPERIMENT["target"] = "target_mispricing_score"
EXPERIMENT["secondary_target"] = "future_excess_return_12m"

ML_CONFIG["feature_set_type"] = "fundamental_plus_market"
ML_CONFIG["classification_labels"] = ["NOT_INVESTABLE", "INVESTABLE"]
ML_CONFIG["prediction_horizon_months"] = 12

EXPERIMENT["feature_blocks"] = [
    "dcf_inputs",
    "quality_ratios",
    "valuation_outputs",
    "market_overlay"
]

logger.info("[0.6 bis] Notebook riallineato a company_analysis_with_ml_overlay")
print("analysis_mode      :", EXPERIMENT["analysis_mode"])
print("primary_objective  :", EXPERIMENT["primary_objective"])
print("target             :", EXPERIMENT["target"])
print("secondary_target   :", EXPERIMENT["secondary_target"])
print("feature_set_type   :", ML_CONFIG["feature_set_type"])
print("class_labels       :", ML_CONFIG["classification_labels"])

In [109]:
# 0.4 Output folders e logging

OUTPUT_ROOT = PROJECT_ROOT / "output"
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR  = OUTPUT_ROOT / "tables"
LOGS_DIR    = OUTPUT_ROOT / "logs"

for d in [OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log_file = LOGS_DIR / f"{EXPERIMENT['name']}.log"

logger = logging.getLogger(EXPERIMENT["name"])
logger.setLevel(logging.INFO)
logger.handlers = []

fh = logging.FileHandler(log_file)
fh.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
ch.setFormatter(formatter)

logger.addHandler(fh)
logger.addHandler(ch)

logger.info("Experiment started")

2026-05-12 21:54:25,161 - INFO - Experiment started
INFO:fundamental_valuation_dcf_definitive:Experiment started


In [136]:
# Path canonici riusabili in tutto il notebook
OUTPUT_DIR = OUTPUT_ROOT
TABLES_OUTPUT_DIR = TABLES_DIR
FIGURES_OUTPUT_DIR = FIGURES_DIR
LOGS_OUTPUT_DIR = LOGS_DIR

## 1. 📥 Data Ingestion

In questa sezione carichiamo i dati finanziari direttamente dalla cartella **Database Finanziario** su Google Drive, usando gli ID file per un accesso robusto indipendente dal path di montaggio.

| Cell | What it does | Output |
|------|---------------|--------|
| 1.1  | Monta Drive e scarica i file del Database Finanziario via `gdown` | File locali in `/content/data_db/` |
| 1.2  | Loader intelligente: legge il file con più colonne (package > prices) | `df_panel` pronto per il DCF |
| 1.3  | Carica anche i risk factors per il calcolo del costo del capitale | `df_risk` |
| 1.4  | Data source summary | `Table_1_data_source_summary.csv` |

Il **Database Finanziario** è il data lake centralizzato del progetto: contiene prezzi giornalieri, fattori di rischio e dati fondamentali per l'universo azionario. Il pannello integrato permette di costruire un DCF completo per ogni titolo e periodo senza dover riconciliare manualmente sorgenti diverse.

$$
\text{Panel}_{i,t} = \{\,P_{i,t},\; \text{FCF}_{i,t},\; r^f_t,\; \beta_i,\; \text{ERP}_t\,\}
$$

In [110]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Database Finanziario")

print("Contenuto di MyDrive (prime 30 voci):")
for p in list(base.glob("*"))[:30]:
    print(" -", p.name)

Contenuto di MyDrive (prime 30 voci):
 - Betting Against Beta Original Paper Data.xlsx
 - Betting Against Beta Equity Factors Daily.xlsx
 - Quality Minus Junk Factors Daily.xlsx
 - The Devil in HMLs Details Factors Daily.xlsx
 - Cripto_MinuteTimeSeries
 - europe_italy_market_data
 - europe_stoxx_companies
 - alternative_data
 - Finance Database
 - catalog
 - data_providers
 - alpha_factor_research
 - bayesian_machine_learning
 - deep_learning
 - convolutional_neural_nets
 - autoencoders
 - deep_reinforcement_learning
 - alpha_factor_library
 - data
 - ml_trading_risk_factors_csv
 - ml_trading_market_prices_csv
 - ml_trading_database_package.gsheet


In [111]:
# 1.1 Install dipendenze, monta Drive, scopre i file del Database Finanziario

import subprocess, sys, os, logging, shutil
from pathlib import Path
import numpy as np
import pandas as pd

for pkg in ["gdown", "yfinance", "pandas-datareader", "gspread",
            "google-auth", "openpyxl", "pyarrow", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Monta Drive
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    logger.info("Google Drive montato: /content/drive/MyDrive")
except Exception as e:
    logger.warning(f"Drive mount skipped: {e}")

# Trova DB_ROOT dinamicamente
mydrive = Path("/content/drive/MyDrive")
db_name = next(
    (n for n in os.listdir(str(mydrive))
     if "Database" in n and "Finanziario" in n), None
)
DB_ROOT = mydrive / db_name if db_name else None
logger.info(f"DB_ROOT: {DB_ROOT} | exists: {DB_ROOT.exists() if DB_ROOT else False}")

# Cartella locale di lavoro
DATA_LOCAL = Path("/content/data_db")
DATA_LOCAL.mkdir(parents=True, exist_ok=True)

# Repertorio file nel Database Finanziario
# ml_trading_market_prices_csv  → CSV con prezzi giornalieri
# ml_trading_risk_factors_csv   → CSV con fattori di rischio Fama-French
# ml_trading_database_package   → Google Sheet (accesso via gspread)
FILE_MAP = {
    "market_prices": {
        "drive_name": "ml_trading_market_prices_csv",
        "local"     : DATA_LOCAL / "ml_trading_market_prices.csv",
        "is_gsheet" : False,
        "required"  : True,
    },
    "risk_factors": {
        "drive_name": "ml_trading_risk_factors_csv",
        "local"     : DATA_LOCAL / "ml_trading_risk_factors.csv",
        "is_gsheet" : False,
        "required"  : False,
    },
    "db_package": {
        "drive_name": "ml_trading_database_package",
        "local"     : DATA_LOCAL / "ml_trading_database_package.csv",
        "is_gsheet" : True,
        "required"  : False,
    },
}

# Copia file CSV/Parquet da Drive
for key, cfg in FILE_MAP.items():
    if cfg["is_gsheet"]:
        continue  # gestito in 1.2
    local = cfg["local"]
    if local.exists():
        logger.info(f"[1.1] {key}: già in cache → {local}")
        continue
    if DB_ROOT:
        src = DB_ROOT / cfg["drive_name"]
        for ext in ["", ".csv", ".parquet", ".xlsx"]:
            candidate = Path(str(src) + ext)
            if candidate.exists() and not candidate.suffix == ".gsheet":
                try:
                    shutil.copy2(str(candidate), str(local))
                    logger.info(f"[1.1] {key}: copiato da {candidate.name}")
                    break
                except OSError as e:
                    logger.warning(f"[1.1] {key}: copia fallita ({e})")
        else:
            logger.warning(f"[1.1] {key}: non trovato in DB_ROOT")

EXPERIMENT["db_root"]  = str(DB_ROOT) if DB_ROOT else "N/A"
EXPERIMENT["file_map"] = {k: str(v["local"]) for k, v in FILE_MAP.items()}
EXPERIMENT["data_local"] = str(DATA_LOCAL)

print("=== FILE STATUS ===")
for key, cfg in FILE_MAP.items():
    p = cfg["local"]
    print(f"  {key:15s} | exists={str(p.exists()):5s} | {p.name}")

2026-05-12 21:55:00,986 - INFO - Google Drive montato: /content/drive/MyDrive
INFO:fundamental_valuation_dcf_definitive:Google Drive montato: /content/drive/MyDrive
2026-05-12 21:55:00,993 - INFO - DB_ROOT: /content/drive/MyDrive/Database Finanziario | exists: True
INFO:fundamental_valuation_dcf_definitive:DB_ROOT: /content/drive/MyDrive/Database Finanziario | exists: True
2026-05-12 21:55:00,996 - INFO - [1.1] market_prices: già in cache → /content/data_db/ml_trading_market_prices.csv
INFO:fundamental_valuation_dcf_definitive:[1.1] market_prices: già in cache → /content/data_db/ml_trading_market_prices.csv
2026-05-12 21:55:00,998 - INFO - [1.1] risk_factors: già in cache → /content/data_db/ml_trading_risk_factors.csv
INFO:fundamental_valuation_dcf_definitive:[1.1] risk_factors: già in cache → /content/data_db/ml_trading_risk_factors.csv


=== FILE STATUS ===
  market_prices   | exists=True  | ml_trading_market_prices.csv
  risk_factors    | exists=True  | ml_trading_risk_factors.csv
  db_package      | exists=False | ml_trading_database_package.csv


In [ ]:
# ============================================================
# 1.1 bis - Cache manager + persistence su Drive + manifest
# ============================================================

import os
import json
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

DATA_LOCAL = Path("/content/data_db")
RAW_CACHE = DATA_LOCAL / "raw_cache"
PROCESSED_CACHE = DATA_LOCAL / "processed_cache"
REPORTS_CACHE = DATA_LOCAL / "reports"

for p in [DATA_LOCAL, RAW_CACHE, PROCESSED_CACHE, REPORTS_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

DB_ROOT = Path(EXPERIMENT.get("db_root", "/content/drive/MyDrive/Database Finanziario"))
DB_ROOT = Path(DB_ROOT) if str(DB_ROOT) != "N/A" else None

if DB_ROOT and DB_ROOT.exists():
    DRIVE_RAW_CACHE = DB_ROOT / "raw_cache"
    DRIVE_PROCESSED_CACHE = DB_ROOT / "processed_cache"
    DRIVE_REPORTS = DB_ROOT / "reports"
    for p in [DRIVE_RAW_CACHE, DRIVE_PROCESSED_CACHE, DRIVE_REPORTS]:
        p.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_RAW_CACHE = None
    DRIVE_PROCESSED_CACHE = None
    DRIVE_REPORTS = None

MANIFEST_PATH_LOCAL = DATA_LOCAL / "run_manifest.json"
MANIFEST_PATH_DRIVE = (DB_ROOT / "run_manifest.json") if DB_ROOT and DB_ROOT.exists() else None

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def persist_file(local_path, drive_dir=None, logger=None):
    local_path = Path(local_path)
    if logger is None:
        logger = logging.getLogger(__name__)

    meta = {
        "local_path": str(local_path),
        "exists": local_path.exists(),
        "size_bytes": None,
        "md5": None,
        "drive_path": None,
        "copied_to_drive": False,
        "updated_at": datetime.utcnow().isoformat()
    }

    if not local_path.exists():
        return meta

    meta["size_bytes"] = local_path.stat().st_size
    meta["md5"] = file_md5(local_path)

    if drive_dir is not None:
        drive_dir = Path(drive_dir)
        drive_dir.mkdir(parents=True, exist_ok=True)
        target = drive_dir / local_path.name
        try:
            shutil.copy2(local_path, target)
            meta["drive_path"] = str(target)
            meta["copied_to_drive"] = True
            logger.info(f"[persist] copiato {local_path.name} -> {target}")
        except Exception as e:
            logger.warning(f"[persist] copia fallita per {local_path.name}: {e}")

    return meta

RUN_MANIFEST = {
    "run_timestamp_utc": datetime.utcnow().isoformat(),
    "experiment_name": EXPERIMENT.get("name"),
    "db_root": str(DB_ROOT) if DB_ROOT else None,
    "datasets": {}
}

EXPERIMENT["data_local"] = str(DATA_LOCAL)
EXPERIMENT["raw_cache"] = str(RAW_CACHE)
EXPERIMENT["processed_cache"] = str(PROCESSED_CACHE)
EXPERIMENT["reports_cache"] = str(REPORTS_CACHE)

print("DATA_LOCAL:", DATA_LOCAL)
print("DB_ROOT   :", DB_ROOT)

In [113]:
# 1.2 Carica df_panel: Drive locale → API yfinance → sintetico

import yfinance as yf
from tqdm import tqdm

# Universe predefinito se non carichiamo da file
DEFAULT_TICKERS_SP500_SAMPLE = [
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","BRK-B","JPM","JNJ","V",
    "PG","UNH","HD","MA","DIS","PYPL","ADBE","NFLX","INTC","CSCO",
    "PFE","KO","PEP","ABT","TMO","NKE","MRK","WMT","BAC","XOM",
    "CVX","LLY","ABBV","AVGO","COST","DHR","ACN","TXN","NEE","UNP",
    "MDT","LIN","HON","PM","AMGN","BMY","UPS","SBUX","IBM","GE"
]

def load_prices_from_api(tickers, start, end, logger=None):
    if logger is None:
        logger = logging.getLogger(__name__)
    logger.info(f"[API] Scarico prezzi per {len(tickers)} ticker via yfinance")
    frames = []
    for t in tqdm(tickers, desc="yfinance download"):
        try:
            raw = yf.download(t, start=start, end=end,
                              auto_adjust=True, progress=False)
            if raw.empty:
                continue
            raw = raw[["Close","Volume"]].copy()
            raw.columns = ["adj_close","volume"]
            raw.index.name = "date"
            raw.reset_index(inplace=True)
            raw["ticker"] = t
            frames.append(raw)
        except Exception as e:
            logger.warning(f"[API] {t}: {e}")
    if not frames:
        raise RuntimeError("Nessun dato scaricato da yfinance")
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values(["ticker","date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    logger.info(f"[API] Panel scaricato: shape={df.shape}")
    return df

def load_dataframe_safe(path: Path, logger=None):
    if logger is None:
        logger = logging.getLogger(__name__)
    for candidate in [path,
                      Path(str(path)+".parquet"),
                      Path(str(path)+".csv")]:
        if not candidate.exists():
            continue
        try:
            if candidate.suffix in [".parquet",".pq"]:
                df = pd.read_parquet(candidate)
            elif candidate.suffix == ".csv":
                df = pd.read_csv(candidate, low_memory=False)
            elif candidate.suffix in [".xlsx",".xls"]:
                df = pd.read_excel(candidate)
            else:
                continue
            logger.info(f"Loaded {candidate.name} shape={df.shape}")
            return df, str(candidate)
        except Exception as e:
            logger.warning(f"Errore lettura {candidate}: {e}")
    return None, None

# ── Tentativo 1: file locale da DB ───────────────────────────────────────
df_panel = None
panel_source = None

prices_path = Path(EXPERIMENT["file_map"]["market_prices"])
df_panel, panel_path = load_dataframe_safe(prices_path, logger)

if df_panel is not None and df_panel.shape[1] >= 3:
    panel_source = "database_finanziario_local"
    logger.info(f"[1.2] Panel caricato da file locale: {panel_path}")
else:
    logger.warning("[1.2] File locale non valido o mancante → fallback API yfinance")
    # ── Tentativo 2: API yfinance ─────────────────────────────────────────
    try:
        df_panel = load_prices_from_api(
            tickers=DEFAULT_TICKERS_SP500_SAMPLE,
            start=EXPERIMENT["start_date"],
            end=EXPERIMENT["end_date"],
            logger=logger
        )
        panel_source = "yfinance_api"
        # Salva sul Database Finanziario per uso futuro
        save_path = DATA_LOCAL / "ml_trading_market_prices.csv"
        df_panel.to_csv(save_path, index=False)
        logger.info(f"[1.2] Panel scaricato da API e salvato in {save_path}")
        # Copia anche su Drive per aggiornamenti futuri
        if DB_ROOT and DB_ROOT.exists():
            drive_save = DB_ROOT / "ml_trading_market_prices_csv.csv"
            try:
                shutil.copy2(str(save_path), str(drive_save))
                logger.info(f"[1.2] Panel salvato su Drive: {drive_save}")
            except Exception as e:
                logger.warning(f"[1.2] Salvataggio su Drive fallito: {e}")
    except Exception as e:
        logger.error(f"[1.2] API yfinance fallita: {e}")
        # ── Fallback sintetico ────────────────────────────────────────────
        logger.warning("[1.2] Generazione panel SINTETICO")
        dates  = pd.date_range(EXPERIMENT["start_date"], EXPERIMENT["end_date"], freq="B")
        tickers= ["AAPL_synthetic","MSFT_synthetic","GOOGL_synthetic"]
        rows   = []
        for t in tickers:
            prices = 100 * np.exp(np.cumsum(np.random.normal(0.0003,0.015,len(dates))))
            for i,d in enumerate(dates):
                rows.append({"date":d,"ticker":t,
                             "adj_close":prices[i],"volume":1e6,
                             "adj_close_synthetic":True})
        df_panel = pd.DataFrame(rows)
        panel_source = "synthetic"
        logger.warning("[1.2] Panel sintetico creato")

# Normalizza colonne chiave
for old, new in [
    (next((c for c in df_panel.columns if c.lower() in
           ["datetime","timestamp"]), None), "date"),
    (next((c for c in df_panel.columns if c.lower() in
           ["symbol","asset","stock"]), None), "ticker"),
    (next((c for c in df_panel.columns if c.lower() in
           ["adj close","close","price"]), None), "adj_close"),
]:
    if old and old != new and old in df_panel.columns:
        df_panel.rename(columns={old: new}, inplace=True)

if "date" in df_panel.columns:
    df_panel["date"] = pd.to_datetime(df_panel["date"])
if all(c in df_panel.columns for c in ["ticker","date"]):
    df_panel.sort_values(["ticker","date"], inplace=True)
    df_panel.reset_index(drop=True, inplace=True)

panel_meta = {
    "source"   : panel_source,
    "synthetic": panel_source == "synthetic",
    "path"     : panel_path or "api/synthetic",
    "mode"     : "database_finanziario",
}

logger.info(f"[1.2] df_panel finale: shape={df_panel.shape} | source={panel_source}")
print(f"Shape  : {df_panel.shape}")
print(f"Source : {panel_source}")
print(f"Columns: {df_panel.columns.tolist()}")
print(df_panel.head())

2026-05-12 21:55:01,475 - INFO - Loaded ml_trading_market_prices.csv shape=(140823, 11)
INFO:fundamental_valuation_dcf_definitive:Loaded ml_trading_market_prices.csv shape=(140823, 11)
2026-05-12 21:55:01,478 - INFO - [1.2] Panel caricato da file locale: /content/data_db/ml_trading_market_prices.csv
INFO:fundamental_valuation_dcf_definitive:[1.2] Panel caricato da file locale: /content/data_db/ml_trading_market_prices.csv
2026-05-12 21:55:01,567 - INFO - [1.2] df_panel finale: shape=(140823, 11) | source=database_finanziario_local
INFO:fundamental_valuation_dcf_definitive:[1.2] df_panel finale: shape=(140823, 11) | source=database_finanziario_local


Shape  : (140823, 11)
Source : database_finanziario_local
Columns: ['date', 'ticker', 'open', 'high', 'low', 'adj_close', 'adjusted', 'volume', 'source', 'source_symbol', 'asset_class']
        date  ticker  open  high  low  adj_close  adjusted  volume  \
0 2007-01-02  A2A.MI   NaN   NaN  NaN        NaN  1.499469     NaN   
1 2007-01-03  A2A.MI   NaN   NaN  NaN        NaN  1.500938     NaN   
2 2007-01-04  A2A.MI   NaN   NaN  NaN        NaN  1.486251     NaN   
3 2007-01-05  A2A.MI   NaN   NaN  NaN        NaN  1.468628     NaN   
4 2007-01-08  A2A.MI   NaN   NaN  NaN        NaN  1.461285     NaN   

                             source source_symbol asset_class  
0  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
1  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
2  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
3  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
4  export_python:FTSEMIB_prices.csv        A2A.MI      equity  


In [114]:
def choose_best_candidate(candidates):
    if not candidates:
        return None

    preferred_keywords = [
        "feature_engineering_factor_panel",
        "alpha_factor_panel",
        "equity_feature_panel",
        "boosting_model_panel",
        "linear_model_panel",
        "market_prices",
        "clean_panel",
    ]

    excluded_keywords = [
        "backtest",
        "weights",
        "unsupervised",
        "eigen_portfolio",
        "efficient_frontier",
        "constituents",
    ]

    scored = []

    for path in candidates:
        name = path.name.lower()
        score = 0

        # penalizza file non adatti
        for kw in excluded_keywords:
            if kw in name:
                score -= 100

        # premia file panel/factor adatti
        for i, kw in enumerate(preferred_keywords[::-1], start=1):
            if kw in name:
                score += i * 20

        # preferenza formati
        if path.suffix.lower() == ".parquet":
            score += 10
        elif path.suffix.lower() == ".csv":
            score += 5
        elif path.suffix.lower() in [".xlsx", ".xls"]:
            score += 2

        scored.append((score, path))

    scored = sorted(scored, key=lambda x: (-x[0], str(x[1])))
    return scored[0][1]

In [115]:
# ============================================================
# 1.2 bis - Universe builder da Yahoo: STOXX50 + SP500 + FTSEMIB
# ============================================================

import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

UNIVERSE_CONFIG = {
    "use_yahoo_universe_builder": True,
    "include_sp500": True,
    "include_stoxx50": True,
    "include_ftsemib": True,
    "max_download_threads": 8,
}

EXPERIMENT["universe_config"] = UNIVERSE_CONFIG


def normalize_yahoo_ticker(t):
    if pd.isna(t):
        return None
    t = str(t).strip().upper()
    t = t.replace("/", "-")
    return t


def get_sp500_constituents():
    urls = [
        "https://pkgstore.datahub.io/core/s-and-p-500-companies/constituents_csv/data/constituents_csv.csv",
        "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv",
    ]

    last_err = None
    for url in urls:
        try:
            df = pd.read_csv(url)
            break
        except Exception as e:
            last_err = e
            df = None

    if df is None or df.empty:
        raise ValueError(f"Impossibile scaricare S&P 500 da DataHub: {last_err}")

    rename_map = {
        "Symbol": "ticker",
        "Name": "company_name",
        "Security": "company_name",
        "Sector": "sector",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "industry",
        "Sub-Industry": "industry",
    }
    df = df.rename(columns=rename_map).copy()

    required = ["ticker"]
    for c in required:
        if c not in df.columns:
            raise ValueError("Colonna ticker non trovata nel dataset S&P 500")

    if "company_name" not in df.columns:
        df["company_name"] = None
    if "sector" not in df.columns:
        df["sector"] = None
    if "industry" not in df.columns:
        df["industry"] = None

    df["ticker"] = df["ticker"].astype(str).map(normalize_yahoo_ticker)
    df["index_name"] = "SP500"

    return df[["ticker", "company_name", "sector", "industry", "index_name"]]

def get_stoxx50_constituents():
    stoxx50 = [
        "ADS.DE","ADYEN.AS","AIR.PA","ALV.DE","ASML.AS","ABI.BR","BAS.DE","BAYN.DE",
        "BBVA.MC","BNP.PA","CS.PA","DAI.DE","DB1.DE","DG.PA","DHL.DE","ENEL.MI",
        "ENI.MI","IBE.MC","IFX.DE","INGA.AS","ISP.MI","KER.PA","LIN.DE","MC.PA",
        "MUV2.DE","NOKIA.HE","ORA.PA","PHIA.AS","SAF.PA","SAN.MC","SAN.PA","SAP.DE",
        "SCHN.SW","SIE.DE","STLAM.MI","SU.PA","TTE.PA","UCG.MI","ULVR.L","VOW3.DE",
        "AI.PA","ANX.MC","BAMI.MI","BMW.DE","CRH.L","DTE.DE","EL.PA","RMS.PA",
        "RI.PA","PRX.AS"
    ]

    df = pd.DataFrame({"ticker": stoxx50})
    df["ticker"] = df["ticker"].map(normalize_yahoo_ticker)
    df["company_name"] = None
    df["sector"] = None
    df["industry"] = None
    df["index_name"] = "EURO_STOXX_50"
    return df[["ticker", "company_name", "sector", "industry", "index_name"]]

def get_ftsemib_constituents():
    manual_ftsemib = [
        "A2A.MI","AMP.MI","AZM.MI","BMPS.MI","BAMI.MI","BGN.MI","BMED.MI","BREB.MI",
        "BZU.MI","CPR.MI","DIA.MI","ENEL.MI","ENI.MI","FBK.MI","FER.MI","G.MI",
        "HER.MI","IG.MI","INW.MI","IP.MI","ISP.MI","IVG.MI","LDO.MI","MB.MI",
        "MONC.MI","MPS.MI","NEXI.MI","PIRC.MI","PST.MI","PRY.MI","RACE.MI","REC.MI",
        "SFER.MI","SRG.MI","STLAM.MI","STMMI.MI","TEN.MI","TIT.MI","TRN.MI","UCG.MI"
    ]
    df = pd.DataFrame({"ticker": manual_ftsemib})
    df["ticker"] = df["ticker"].map(normalize_yahoo_ticker)
    df["company_name"] = None
    df["sector"] = None
    df["industry"] = None
    df["index_name"] = "FTSEMIB"
    return df[["ticker", "company_name", "sector", "industry", "index_name"]]


universe_parts = []

for loader_name, loader_fn in [
    ("SP500", get_sp500_constituents),
    ("EURO STOXX 50", get_stoxx50_constituents),
    ("FTSEMIB", get_ftsemib_constituents),
]:
    try:
        df_idx = loader_fn()
        universe_parts.append(df_idx)
        logger.info(f"[Universe] {loader_name} constituents caricati: {len(df_idx)}")
    except Exception as e:
        logger.warning(f"[Universe] {loader_name} non disponibile: {e}")

if not universe_parts:
    raise RuntimeError("Nessun universo disponibile")

df_universe = pd.concat(universe_parts, ignore_index=True)
df_universe = df_universe.dropna(subset=["ticker"]).copy()
df_universe["ticker"] = df_universe["ticker"].map(normalize_yahoo_ticker)
df_universe = df_universe.drop_duplicates(subset=["ticker"]).reset_index(drop=True)

EXPERIMENT["universe_size"] = int(len(df_universe))
EXPERIMENT["universe_indices"] = sorted(df_universe["index_name"].dropna().unique().tolist())

print("Universe size:", len(df_universe))
print("Indices:", EXPERIMENT["universe_indices"])
display(df_universe.groupby("index_name")["ticker"].count().reset_index(name="n_tickers"))
display(df_universe.head())

2026-05-12 21:55:05,037 - INFO - [Universe] SP500 constituents caricati: 503
INFO:fundamental_valuation_dcf_definitive:[Universe] SP500 constituents caricati: 503
2026-05-12 21:55:05,042 - INFO - [Universe] EURO STOXX 50 constituents caricati: 50
INFO:fundamental_valuation_dcf_definitive:[Universe] EURO STOXX 50 constituents caricati: 50
2026-05-12 21:55:05,048 - INFO - [Universe] FTSEMIB constituents caricati: 40
INFO:fundamental_valuation_dcf_definitive:[Universe] FTSEMIB constituents caricati: 40


Universe size: 587
Indices: ['EURO_STOXX_50', 'FTSEMIB', 'SP500']


,index_name,n_tickers
0,EURO_STOXX_50,50
1,FTSEMIB,34
2,SP500,503


,ticker,company_name,sector,industry,index_name
0,MMM,3M,Industrials,Industrial Conglomerates,SP500
1,AOS,A. O. Smith,Industrials,Building Products,SP500
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,SP500
3,ABBV,AbbVie,Health Care,Biotechnology,SP500
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,SP500


In [ ]:
# ============================================================
# 1.2 ter - Persist universe constituents
# ============================================================

universe_local_csv = PROCESSED_CACHE / "universe_constituents_master.csv"
universe_local_parquet = PROCESSED_CACHE / "universe_constituents_master.parquet"
universe_by_index_csv = PROCESSED_CACHE / "universe_constituents_by_index_counts.csv"

df_universe.to_csv(universe_local_csv, index=False)
df_universe.to_parquet(universe_local_parquet, index=False)

df_universe_counts = (
    df_universe.groupby("index_name")["ticker"]
    .nunique()
    .reset_index(name="n_tickers")
)
df_universe_counts.to_csv(universe_by_index_csv, index=False)

RUN_MANIFEST["datasets"]["universe_constituents_master_csv"] = persist_file(
    universe_local_csv, DRIVE_PROCESSED_CACHE, logger
)
RUN_MANIFEST["datasets"]["universe_constituents_master_parquet"] = persist_file(
    universe_local_parquet, DRIVE_PROCESSED_CACHE, logger
)
RUN_MANIFEST["datasets"]["universe_constituents_by_index_counts_csv"] = persist_file(
    universe_by_index_csv, DRIVE_REPORTS, logger
)

print("Universe constituents salvati in cache locale e Drive.")
display(df_universe_counts)

In [116]:
# ============================================================
# 1.3 bis - Fundamentals mode: local -> API -> graceful fallback
# Fix: supporta sia FMP_API_KEY sia fmp_api_key
# ============================================================

from pathlib import Path
import os

FUNDAMENTALS_OUTPUT_PATH = Path("/content/data_db/fundamentals_api_panel.csv")
EXPERIMENT["fundamentals_output_path"] = str(FUNDAMENTALS_OUTPUT_PATH)

fundamentals_mode = "local_if_exists_else_api"
EXPERIMENT["fundamentals_mode"] = fundamentals_mode


def get_colab_secret_flexible(secret_names, logger=None):
    if logger is None:
        logger = logging.getLogger(__name__)

    try:
        from google.colab import userdata
    except Exception as e:
        logger.warning(f"google.colab.userdata non disponibile: {e}")
        return None, None

    for secret_name in secret_names:
        try:
            value = userdata.get(secret_name)
            if value and str(value).strip():
                logger.info(f"Secret trovato in Colab: {secret_name}")
                return value, secret_name
        except Exception as e:
            logger.warning(f"Secret {secret_name} non disponibile: {e}")

    return None, None


FMP_API_KEY, FMP_SECRET_NAME = get_colab_secret_flexible(
    ["FMP_API_KEY", "fmp_api_key"],
    logger=logger
)

if not FMP_API_KEY:
    for env_name in ["FMP_API_KEY", "fmp_api_key"]:
        env_value = os.getenv(env_name)
        if env_value and env_value.strip():
            FMP_API_KEY = env_value
            FMP_SECRET_NAME = f"env:{env_name}"
            logger.info(f"FMP API key trovata in variabile ambiente: {env_name}")
            break

local_fundamentals_exists = FUNDAMENTALS_OUTPUT_PATH.exists()

EXPERIMENT["fundamentals_available"] = False
EXPERIMENT["fundamentals_source"] = None
EXPERIMENT["fmp_secret_name_used"] = FMP_SECRET_NAME
EXPERIMENT["fmp_api_enabled"] = bool(FMP_API_KEY)

print("fundamentals_mode:", fundamentals_mode)
print("local fundamentals exists:", local_fundamentals_exists)
print("FMP_API_KEY available:", bool(FMP_API_KEY))
print("secret source:", FMP_SECRET_NAME)

2026-05-12 21:55:05,584 - WARNING - Secret FMP_API_KEY non disponibile: Secret FMP_API_KEY does not exist.
2026-05-12 21:55:06,118 - INFO - Secret trovato in Colab: fmp_api_key
INFO:fundamental_valuation_dcf_definitive:Secret trovato in Colab: fmp_api_key


fundamentals_mode: local_if_exists_else_api
local fundamentals exists: False
FMP_API_KEY available: True
secret source: fmp_api_key


In [117]:
# 1.4 Download fondamentali: yfinance primario → FMP US fallback → sintetico
# Salva sempre su Drive per aggiornamenti futuri

import yfinance as yf
import requests
import time
from tqdm import tqdm

FUNDAMENTALS_OUTPUT_PATH = DATA_LOCAL / "ml_trading_fundamentals.parquet"
FUNDAMENTALS_DRIVE_PATH  = (DB_ROOT / "ml_trading_fundamentals.parquet"
                             if DB_ROOT and DB_ROOT.exists() else None)

# Parole chiave che identificano file NON-fondamentali da escludere
EXCLUDE_KEYWORDS = [
    "factor", "junk", "momentum", "value", "AQR", "FF", "fama",
    "betas", "returns", "backtest", "eigen", "weights", "signal",
    "portfolio", "alpha", "beta", "risk_factor", "Junk", "Quality"
]

def is_fundamental_file(path: Path) -> bool:
    name = path.name.lower()
    return not any(kw.lower() in name for kw in EXCLUDE_KEYWORDS)

df_fund     = pd.DataFrame()
fund_source = None

# ── Tentativo 0: cache locale già salvata ───────────────────────────────
if FUNDAMENTALS_OUTPUT_PATH.exists():
    try:
        df_fund = pd.read_parquet(FUNDAMENTALS_OUTPUT_PATH)
        if "ticker" in df_fund.columns and df_fund.shape[1] > 5:
            fund_source = "local_cache"
            logger.info(f"[1.4] Cache locale: {df_fund.shape}")
        else:
            df_fund = pd.DataFrame()
    except Exception as e:
        logger.warning(f"[1.4] Cache locale corrotta: {e}")
        df_fund = pd.DataFrame()

# ── Tentativo 1: Finance Database su Drive (solo file fondamentali veri) ─
if df_fund.empty and DB_ROOT and DB_ROOT.exists():
    fin_db = DB_ROOT / "Finance Database"
    if fin_db.exists():
        candidates = [
            p for p in (list(fin_db.glob("*.parquet")) +
                        list(fin_db.glob("*.xlsx")) +
                        list(fin_db.glob("*.csv")))
            if is_fundamental_file(p)
        ]
        logger.info(f"[1.4] Finance Database: {len(candidates)} file fondamentali candidati")
        for p in sorted(candidates, key=lambda x: x.stat().st_size, reverse=True):
            logger.info(f"  candidato: {p.name} ({p.stat().st_size/1e6:.1f} MB)")
        # Cerca file con colonne tipo revenue, net_income, ecc.
        REQUIRED_COLS = {"revenue","net_income","total_assets","ebit",
                         "Revenue","NetIncome","TotalAssets"}
        for p in sorted(candidates, key=lambda x: x.stat().st_size, reverse=True):
            try:
                if p.suffix == ".parquet":
                    tmp = pd.read_parquet(p)
                elif p.suffix in [".xlsx",".xls"]:
                    tmp = pd.read_excel(p, nrows=5)
                else:
                    tmp = pd.read_csv(p, nrows=5)
                cols_lower = {c.lower() for c in tmp.columns}
                if REQUIRED_COLS & {c.lower() for c in tmp.columns} or \
                   any(k in cols_lower for k in ["revenue","net_income","assets","ebit"]):
                    # Ricarica completo
                    if p.suffix == ".parquet":
                        df_fund = pd.read_parquet(p)
                    elif p.suffix in [".xlsx",".xls"]:
                        df_fund = pd.read_excel(p)
                    else:
                        df_fund = pd.read_csv(p, low_memory=False)
                    fund_source = "finance_database_drive"
                    logger.info(f"[1.4] Fondamentali da Drive: {p.name} {df_fund.shape}")
                    break
            except Exception as e:
                logger.warning(f"[1.4] {p.name}: {e}")

# ── Tentativo 2: yfinance (gratuito, supporta .MI .PA .DE .L ecc.) ──────
if df_fund.empty:
    logger.info(f"[1.4] yfinance: scarico fondamentali per {len(selected_tickers)} ticker")
    fund_rows = []
    failed    = []

    for ticker in tqdm(selected_tickers, desc="yfinance fundamentals"):
        try:
            info = yf.Ticker(ticker).info
            if not info or len(info) < 5:
                failed.append(ticker)
                continue
            row = {
                "ticker"              : ticker,
                "date_updated"        : pd.Timestamp.today().normalize(),
                # P&L
                "revenue"             : info.get("totalRevenue"),
                "gross_profit"        : info.get("grossProfits"),
                "ebitda"              : info.get("ebitda"),
                "net_income"          : info.get("netIncomeToCommon"),
                "operating_cash_flow" : info.get("operatingCashflow"),
                "free_cash_flow"      : info.get("freeCashflow"),
                "capex"               : info.get("capitalExpenditures"),
                # Balance Sheet
                "total_assets"        : info.get("totalAssets"),
                "total_debt"          : info.get("totalDebt"),
                "total_equity"        : info.get("bookValue"),
                "cash"                : info.get("totalCash"),
                # Ratios
                "pe_ratio"            : info.get("trailingPE"),
                "forward_pe"          : info.get("forwardPE"),
                "pb_ratio"            : info.get("priceToBook"),
                "ps_ratio"            : info.get("priceToSalesTrailing12Months"),
                "ev_ebitda"           : info.get("enterpriseToEbitda"),
                "enterprise_value"    : info.get("enterpriseValue"),
                "market_cap"          : info.get("marketCap"),
                # Per share
                "eps_ttm"             : info.get("trailingEps"),
                "eps_forward"         : info.get("forwardEps"),
                "book_value_per_share": info.get("bookValue"),
                "dividend_yield"      : info.get("dividendYield"),
                # Growth
                "revenue_growth"      : info.get("revenueGrowth"),
                "earnings_growth"     : info.get("earningsGrowth"),
                # Margins
                "profit_margin"       : info.get("profitMargins"),
                "operating_margin"    : info.get("operatingMargins"),
                "gross_margin"        : info.get("grossMargins"),
                # Risk & Meta
                "beta"                : info.get("beta"),
                "shares_outstanding"  : info.get("sharesOutstanding"),
                "float_shares"        : info.get("floatShares"),
                "sector"              : info.get("sector"),
                "industry"            : info.get("industry"),
                "country"             : info.get("country"),
                "currency"            : info.get("currency"),
                "exchange"            : info.get("exchange"),
                "short_name"          : info.get("shortName"),
                "long_name"           : info.get("longName"),
            }
            fund_rows.append(row)
            time.sleep(0.1)
        except Exception as e:
            logger.warning(f"[1.4] yfinance {ticker}: {e}")
            failed.append(ticker)

    if fund_rows:
        df_fund     = pd.DataFrame(fund_rows)
        fund_source = "yfinance_api"
        logger.info(f"[1.4] yfinance: {df_fund.shape} | failed={len(failed)}")

        # Salva locale
        df_fund.to_parquet(FUNDAMENTALS_OUTPUT_PATH, index=False)
        logger.info(f"[1.4] Salvato locale: {FUNDAMENTALS_OUTPUT_PATH}")

        # Salva su Drive per uso futuro
        if FUNDAMENTALS_DRIVE_PATH:
            try:
                df_fund.to_parquet(str(FUNDAMENTALS_DRIVE_PATH), index=False)
                logger.info(f"[1.4] Salvato su Drive: {FUNDAMENTALS_DRIVE_PATH.name}")
            except Exception as e:
                logger.warning(f"[1.4] Salvataggio Drive: {e}")
    else:
        logger.warning("[1.4] yfinance: nessun dato scaricato")

# ── Tentativo 3: FMP solo per ticker US puri (no .MI .PA .DE) ───────────
if df_fund.empty and FMP_API_KEY:
    us_tickers = [t for t in selected_tickers if "." not in t][:20]
    logger.info(f"[1.4] FMP US-only per {len(us_tickers)} ticker")
    fmp_rows = []
    for ticker in tqdm(us_tickers, desc="FMP US"):
        try:
            url = (f"https://financialmodelingprep.com/api/v3/"
                   f"profile/{ticker}?apikey={FMP_API_KEY}")
            r   = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                if data:
                    d = data[0]
                    fmp_rows.append({
                        "ticker"      : ticker,
                        "date_updated": pd.Timestamp.today().normalize(),
                        "market_cap"  : d.get("mktCap"),
                        "beta"        : d.get("beta"),
                        "ev"          : d.get("dcf"),
                        "sector"      : d.get("sector"),
                        "industry"    : d.get("industry"),
                        "country"     : d.get("country"),
                        "currency"    : d.get("currency"),
                    })
            time.sleep(0.2)
        except Exception as e:
            logger.warning(f"[1.4] FMP {ticker}: {e}")

    if fmp_rows:
        df_fund     = pd.DataFrame(fmp_rows)
        fund_source = "fmp_us_only"
        df_fund.to_parquet(FUNDAMENTALS_OUTPUT_PATH, index=False)
        logger.info(f"[1.4] FMP US-only: {df_fund.shape}")

# ── Fallback finale: fondamentali sintetici ──────────────────────────────
if df_fund.empty:
    logger.warning("[1.4] FALLBACK SINTETICO attivato")
    np.random.seed(42)
    tlist = selected_tickers or ["AAPL","MSFT","GOOGL"]
    rows_syn = [{
        "ticker"              : t,
        "date_updated"        : pd.Timestamp.today().normalize(),
        "revenue"             : np.random.uniform(1e9, 1e11),
        "net_income"          : np.random.uniform(1e8, 1e10),
        "total_assets"        : np.random.uniform(5e9, 5e11),
        "total_debt"          : np.random.uniform(1e9, 1e11),
        "total_equity"        : np.random.uniform(2e9, 2e11),
        "free_cash_flow"      : np.random.uniform(5e8, 5e10),
        "operating_cash_flow" : np.random.uniform(5e8, 5e10),
        "ebitda"              : np.random.uniform(1e9, 1e11),
        "beta"                : np.random.uniform(0.5, 2.0),
        "market_cap"          : np.random.uniform(1e10, 1e12),
        "profit_margin"       : np.random.uniform(0.02, 0.35),
        "sector"              : "Unknown_synthetic",
        "country"             : "Unknown_synthetic",
        "revenue_synthetic"   : True,
    } for t in tlist]
    df_fund     = pd.DataFrame(rows_syn)
    fund_source = "synthetic"

EXPERIMENT["fundamentals_available"] = not df_fund.empty
EXPERIMENT["fundamentals_source"]    = fund_source

print(f"fundamentals_available : {EXPERIMENT['fundamentals_available']}")
print(f"fundamentals_source    : {fund_source}")
print(f"df_fund shape          : {df_fund.shape}")
print(f"df_fund columns        : {df_fund.columns.tolist()}")
print()
print(df_fund.head(3).to_string())

2026-05-12 21:55:06,222 - INFO - [1.4] Finance Database: 143 file fondamentali candidati
INFO:fundamental_valuation_dcf_definitive:[1.4] Finance Database: 143 file fondamentali candidati
2026-05-12 21:55:06,264 - INFO -   candidato: M&I Multiple Foundamental Items.xlsx (24.6 MB)
INFO:fundamental_valuation_dcf_definitive:  candidato: M&I Multiple Foundamental Items.xlsx (24.6 MB)
2026-05-12 21:55:06,269 - INFO -   candidato: 118 Stocks TimeSeries 10y.xlsx (21.7 MB)
INFO:fundamental_valuation_dcf_definitive:  candidato: 118 Stocks TimeSeries 10y.xlsx (21.7 MB)
2026-05-12 21:55:06,275 - INFO -   candidato: Entire Universe Clean Time Series.xlsx (17.5 MB)
INFO:fundamental_valuation_dcf_definitive:  candidato: Entire Universe Clean Time Series.xlsx (17.5 MB)
2026-05-12 21:55:06,282 - INFO -   candidato: Database Stock Universe Clean .xlsx (5.9 MB)
INFO:fundamental_valuation_dcf_definitive:  candidato: Database Stock Universe Clean .xlsx (5.9 MB)
2026-05-12 21:55:06,288 - INFO -   candidato:

fundamentals_available : True
fundamentals_source    : finance_database_drive
df_fund shape          : (118, 69)
df_fund columns        : ['Ticker', 'Country', 'Exchange', 'Industry', 'Sector', 'Market Cap', 'Enterprise Value', 'Trailing P/E LTM', 'Forward P/E FY1', 'Price/Sales LTM', 'EV/EBITDA LTM', 'Price/Cash Flow LTM', 'Price/Book LTM', 'ROE LTM', 'ROA LTM', 'Gross Margin LTM', 'Operating Margin LTM', 'Operating Margin 5 Yr Avg', 'Pretax Margin FY0', 'EBITDA/Equity LTM', 'Debt/Equity Latest', 'Net Debt/EBITDA LTM', 'Cash and Equivalents FY0 (Mil)', 'Current Ratio Latest', 'Quick Ratio Latest', 'Inventory Turns Latest', 'Accts Receivable FY0 (mil)', 'Company Shares ', 'Price To Book Value Per Share (Daily Time Series Ratio)', 'Historic P/E', 'Earnings Per Share - SmartEstimate®', 'Beta Three Year Weekly', 'Book Value Per Share', 'Free Cash Flow', 'P/FCF', 'Sales', 'EBIT', 'EBITDA', 'Net Income', 'Earnings per Share', 'Cash & ST Investments', 'Current Assets - Total', 'Total Assets'

In [ ]:
# ============================================================
# 1.4 ter - Canonicalize fundamentals and preserve non-empty dffund
# ============================================================

logger.info("1.4 ter Canonical fundamentals layer")

if "dffund" not in globals() or dffund is None:
    dffund = pd.DataFrame()

dffund_raw = dffund.copy()

def _std_cols(cols):
    return [str(c).strip() for c in cols]

def _find_col(df_, candidates):
    cmap = {str(c).strip().lower(): c for c in df_.columns}
    for cand in candidates:
        if cand.lower() in cmap:
            return cmap[cand.lower()]
    return None

if not dffund_raw.empty:
    dffund_raw.columns = _std_cols(dffund_raw.columns)

    ticker_col = _find_col(dffund_raw, ["ticker", "symbol", "Ticker", "Symbol"])
    if ticker_col is None:
        raise ValueError("Nessuna colonna ticker trovata in dffund")

    dffund_raw["ticker"] = dffund_raw[ticker_col].astype(str).str.strip().str.upper()

    date_col = _find_col(dffund_raw, [
        "statement_date", "statementDate", "date", "Date",
        "fiscalDateEnding", "fillingDate", "acceptedDate", "dateUpdated"
    ])

    if date_col is not None:
        dffund_raw["statement_date"] = pd.to_datetime(dffund_raw[date_col], errors="coerce")
    else:
        dffund_raw["statement_date"] = pd.NaT

    filling_col = _find_col(dffund_raw, ["fillingDate", "filingDate", "acceptedDate", "dateUpdated"])
    if filling_col is not None:
        dffund_raw["available_date"] = pd.to_datetime(dffund_raw[filling_col], errors="coerce")
    else:
        dffund_raw["available_date"] = dffund_raw["statement_date"] + pd.Timedelta(days=90)

    dffund_raw["available_date"] = pd.to_datetime(dffund_raw["available_date"], errors="coerce")
    dffund_raw = dffund_raw[dffund_raw["ticker"].notna()].copy()

    EXPERIMENT["fundamentals_available"] = not dffund_raw.empty
    EXPERIMENT["fundamentals_source"] = EXPERIMENT.get("fundamentals_source", "canonicalized_existing")
    dffund = dffund_raw.copy()

else:
    logger.warning("1.4 ter dffund è vuoto: notebook continuerà in modalità market-only finché non viene popolato.")
    EXPERIMENT["fundamentals_available"] = False

print("fundamentals_available:", EXPERIMENT["fundamentals_available"])
print("dffund shape:", dffund.shape)
if not dffund.empty:
    display(dffund.head(3))

In [ ]:
# ============================================================
# QA - Notebook data integrity checks
# ============================================================

qa_rows = []

def add_qa(check_name, status, details):
    qa_rows.append({
        "check_name": check_name,
        "status": status,
        "details": details
    })

try:
    add_qa("universe_exists", "PASS" if "df_universe" in globals() and not df_universe.empty else "FAIL",
           f"rows={0 if 'df_universe' not in globals() else len(df_universe)}")

    add_qa("prices_exists", "PASS" if "df_panel" in globals() and not df_panel.empty else "FAIL",
           f"rows={0 if 'df_panel' not in globals() else len(df_panel)}")

    add_qa("risk_exists", "PASS" if "df_risk" in globals() and not df_risk.empty else "WARN",
           f"rows={0 if 'df_risk' not in globals() else len(df_risk)}")

    add_qa("fundamentals_exists", "PASS" if "df_fund" in globals() and not df_fund.empty else "WARN",
           f"rows={0 if 'df_fund' not in globals() else len(df_fund)}")

    if "df_universe" in globals() and "df_panel" in globals() and not df_universe.empty and not df_panel.empty:
        u = set(df_universe["ticker"].dropna().astype(str).unique())
        p = set(df_panel["ticker"].dropna().astype(str).unique())
        covered = len(u & p)
        add_qa(
            "universe_price_coverage",
            "PASS" if covered / max(len(u), 1) >= 0.70 else "WARN",
            f"covered={covered}/{len(u)} ({covered/max(len(u),1):.1%})"
        )

    if "df_universe" in globals() and "df_fund" in globals() and df_fund is not None and not df_fund.empty:
        u = set(df_universe["ticker"].dropna().astype(str).unique())
        f = set(df_fund["ticker"].dropna().astype(str).unique())
        covered = len(u & f)
        add_qa(
            "universe_fundamental_coverage",
            "PASS" if covered / max(len(u), 1) >= 0.40 else "WARN",
            f"covered={covered}/{len(u)} ({covered/max(len(u),1):.1%})"
        )

    if "df_panel" in globals() and not df_panel.empty:
        required_cols = ["date", "ticker"]
        missing = [c for c in required_cols if c not in df_panel.columns]
        add_qa("prices_required_columns", "PASS" if not missing else "FAIL", f"missing={missing}")

    if "df_fund" in globals() and df_fund is not None and not df_fund.empty:
        required_cols_f = ["ticker"]
        missing_f = [c for c in required_cols_f if c not in df_fund.columns]
        add_qa("fundamentals_required_columns", "PASS" if not missing_f else "FAIL", f"missing={missing_f}")

except Exception as e:
    add_qa("qa_runtime", "FAIL", str(e))

df_qa = pd.DataFrame(qa_rows)
qa_csv = REPORTS_CACHE / "qa_notebook_checks.csv"
df_qa.to_csv(qa_csv, index=False)

RUN_MANIFEST["datasets"]["qa_notebook_checks"] = persist_file(
    qa_csv, DRIVE_REPORTS, logger
)

display(df_qa)

In [ ]:
# ============================================================
# Finalize manifest
# ============================================================

with open(MANIFEST_PATH_LOCAL, "w", encoding="utf-8") as f:
    json.dump(RUN_MANIFEST, f, indent=2, ensure_ascii=False)

if MANIFEST_PATH_DRIVE is not None:
    try:
        shutil.copy2(MANIFEST_PATH_LOCAL, MANIFEST_PATH_DRIVE)
        logger.info(f"Manifest copiato su Drive: {MANIFEST_PATH_DRIVE}")
    except Exception as e:
        logger.warning(f"Manifest copy failed: {e}")

print("Manifest salvato:", MANIFEST_PATH_LOCAL)
if MANIFEST_PATH_DRIVE:
    print("Manifest Drive :", MANIFEST_PATH_DRIVE)

## 2. 🧹 Cleaning & Alignment

In questa sezione normalizziamo il pannello, eliminiamo look-ahead bias e allineamo prezzi con fondamentali per costruire un panel pronto per il DCF.

| Cell | What it does | Output |
|------|---------------|--------|
| 2.1  | Normalizza tipi, ordina per (ticker, date) | `df` standardizzato |
| 2.2  | Shift t−1 dei fondamentali (zero look-ahead) | `df_dcf_ready` |
| 2.3  | Merge con fondamentali yfinance | `df_dcf_merged` |
| 2.4  | Filtri: periodo, duplicati, prezzi nulli | `df_dcf_filtered` |
| 2.5  | Cleaning report | `Table_2_cleaning_report.csv` |

Lo shift temporale garantisce che ogni feature fondamentale usata al tempo $t$ sia effettivamente disponibile all'investitore prima di $t$:

$$\tilde{X}_{i,t} = X_{i,t-1}$$

Il merge prezzi–fondamentali usa un **as-of join** per associare a ogni data di prezzo il fondamentale più recente disponibile, evitando il look-ahead tipico dei join esatti su date di bilancio.

In [148]:
# 2.1 Normalizzazione colonne chiave

df = df_panel.copy()

# Rileva automaticamente colonne chiave
date_col   = next((c for c in df.columns if c.lower() in ["date","datetime","timestamp"]), None)
ticker_col = next((c for c in df.columns if c.lower() in ["ticker","symbol","asset","stock"]), None)

# --- MODIFIED LOGIC FOR PRICE COLUMN SELECTION ---
# Prioritizza 'adjusted' se contiene valori non-NaN, altrimenti 'adj_close', poi 'close'
price_col_candidates = ["adjusted", "adj_close", "adj close", "close"]
price_col = None
for candidate in price_col_candidates:
    found_col = next((c for c in df.columns if c.lower() == candidate), None)
    if found_col is not None and df[found_col].notna().any(): # Check if it exists and has non-NaN values
        price_col = found_col
        break

if date_col is None:
    raise ValueError("Nessuna colonna data trovata. Colonne disponibili: " + str(df.columns.tolist()))
if ticker_col is None:
    raise ValueError("Nessuna colonna ticker trovata. Colonne disponibili: " + str(df.columns.tolist()))
if price_col is None:
    # Fallback if no suitable price column found after checking candidates
    logger.warning("Nessuna colonna prezzo valida trovata. Useremo la prima colonna numerica disponibile che non sia 'volume'.")
    numeric_cols = df.select_dtypes(include="number").columns
    price_col_fallback = next((c for c in numeric_cols if c.lower() not in ["volume", "id"]), None)
    if price_col_fallback:
        price_col = price_col_fallback
    else:
        raise ValueError("Nessuna colonna prezzo adatta trovata.")

# Standardize the column names
# First, rename date_col and ticker_col
df.rename(columns={date_col: "date", ticker_col: "ticker"}, inplace=True)

# Now handle the price column:
# If the identified price_col is not already 'adj_close', overwrite/create 'adj_close' with its values.
# Then, if the original price_col is different from 'adj_close', drop it to avoid duplicates.
if price_col.lower() != "adj_close" and "adj_close" in df.columns:
    # If the chosen price_col is not the existing 'adj_close' and 'adj_close' exists,
    # we assume the existing 'adj_close' is the problematic one and overwrite it.
    df["adj_close"] = df[price_col]
    if price_col in df.columns and price_col != "adj_close": # If price_col is different and still exists
        df.drop(columns=[price_col], inplace=True) # Drop the redundant price_col
elif price_col.lower() != "adj_close": # If chosen price_col is not adj_close and no adj_close exists, just rename
    df.rename(columns={price_col: "adj_close"}, inplace=True)

# Ensure 'date' column is datetime and sort
df["date"] = pd.to_datetime(df["date"])
df.sort_values(["ticker", "date"], inplace=True)
df.reset_index(drop=True, inplace=True)

logger.info(f"2.1 | shape={df.shape} | date_col=date | ticker_col=ticker | price_col={price_col} (standardized to adj_close)")
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"N tickers: {df['ticker'].nunique()}")
print(df[["date","ticker","adj_close"]].head())

2026-05-12 23:58:25,261 - INFO - 2.1 | shape=(140823, 11) | date_col=date | ticker_col=ticker | price_col=adjusted (standardized to adj_close)
INFO:fundamental_valuation_dcf_definitive:2.1 | shape=(140823, 11) | date_col=date | ticker_col=ticker | price_col=adjusted (standardized to adj_close)


Shape: (140823, 11)
Date range: 2007-01-02 → 2021-05-16
N tickers: 64
        date  ticker  adj_close
0 2007-01-02  A2A.MI   1.499469
1 2007-01-03  A2A.MI   1.500938
2 2007-01-04  A2A.MI   1.486251
3 2007-01-05  A2A.MI   1.468628
4 2007-01-08  A2A.MI   1.461285


In [126]:
print("df_fund exists:", "df_fund" in globals())
if "df_fund" in globals() and df_fund is not None:
    print("df_fund shape:", df_fund.shape)
    print("df_fund columns:", df_fund.columns.tolist()[:40])
    if not df_fund.empty:
        display(df_fund.head())

df_fund exists: True
df_fund shape: (0, 0)
df_fund columns: []


In [127]:
if EXPERIMENT["merge_mode"] == "market_only":
    EXPERIMENT["ML_CONFIG"]["feature_set_type"] = "market_only"
    logger.warning("[2.3] ML_CONFIG aggiornato automaticamente a market_only")

2026-05-12 22:12:31,766 - WARNING - [2.3] ML_CONFIG aggiornato automaticamente a market_only


In [129]:
# ── INJECT SECTIONS 3-14 ──────────────────────────────────────────────────────
import nbformat, json, os, glob
from pathlib import Path

# Trova il file .ipynb corrente in /content/
nb_paths = glob.glob('/root/.local/share/jupyter/runtime/*.json')
# Approccio diretto: scrivi le celle come file python e poi le aggiungiamo
# Usiamo un approccio diverso: creiamo un file con il codice di tutte le sezioni

SECTIONS_CODE = '''
import nbformat
from pathlib import Path
import glob, os

# Trova il notebook
nb_files = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)
nb_files += glob.glob('/root/**/*.ipynb', recursive=True)
print("Notebooks trovati:", [f for f in nb_files if 'Untitled2' in f or 'fundamental' in f.lower() or 'dcf' in f.lower()])
'''
print("Cercando il notebook...")
import glob
nb_files = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)
nb_local = glob.glob('/root/**/*.ipynb', recursive=True)
all_nb = nb_files + nb_local
print("Notebooks su Drive:", len(nb_files))
print("Notebooks locali:", nb_local[:5])
for f in all_nb:
    if 'Untitled2' in f or 'DCF' in f or 'dcf' in f or 'fundamental' in f.lower():
        print(" >> MATCH:", f)

Cercando il notebook...
Notebooks su Drive: 170
Notebooks locali: []
 >> MATCH: /content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_data/00_market_fundamental_data_definitive.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_data/01_NASDAQ_TotalView-ITCH_Order_Book/01_parse_itch_order_flow_messages.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_data/01_NASDAQ_TotalView-ITCH_Order_Book/02_rebuild_nasdaq_order_book.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_data/01_NASDAQ_TotalView-ITCH_Order_Book/03_normalize_tick_data.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_data/02_algoseek_intraday/algoseek_minute_data.ipynb
 >> MATCH: /content/drive/MyDrive/machine-learning-for-trading/02_market_and_fundamental_da

## 3. 🔧 Feature Engineering

> **Obiettivo**: costruire il set di feature tecniche, di momentum e fondamentali
> che alimenta tutti i modelli di valutazione e i modelli ML.

| Cell | What it does | Output |
|------|-------------|--------|
| 3.1 | Returns multi-orizzonte + volatilità realizzata | `df` aggiornato |
| 3.2 | RSI, MACD, Bollinger %B, Price/SMA | `df` aggiornato |
| 3.3 | Momentum cross-sectional (rank, z-score, 12-1) | `df` aggiornato |
| 3.4 | Feature fondamentali UCG (P/E, EV/EBITDA, ROE…) | `df` aggiornato |
| 3.5 | Winsorize (1–99 pct) + missingness table + save | `df_feat`, `Table_III_missingness.csv` |

**Intuizione economica**  
Il momentum a 12-1 mesi cattura la persistenza documentata da Jegadeesh & Titman (1993):

$$\text{Mom}_{12,1} = r_{t-252,t-21}$$

La volatilità realizzata annualizzata su finestra $T$:

$$\sigma_T = \sqrt{252} \cdot \sqrt{\frac{1}{T}\sum_{i=1}^{T} r_i^2}$$

I multipli fondamentali catturano il premio di valore (Fama & French, 1992):

$$r_{t+1}^e = \alpha + \beta_1\,\text{Mom}_{12,1} + \beta_2\,\frac{B}{M} + \beta_3\,\sigma_{21} + \varepsilon$$

In [ ]:
import nbformat
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell
import copy, json

NB_PATH = '/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb'

with open(NB_PATH, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

print(f"Notebook caricato: {len(nb.cells)} celle esistenti")

# ─────────────────────────────────────────────────────────────────
# DEFINIZIONE NUOVE CELLE SEZIONI 3-14
# ─────────────────────────────────────────────────────────────────

new_cells = []

# ══════════════════════════════════════════════════════════════════
# SEZIONE 3 - FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════
new_cells.append(new_markdown_cell("""## 3. 🔧 Feature Engineering

> **Obiettivo**: costruire il set di feature tecniche, di momentum e fondamentali che alimenta i modelli predittivi.

| Cell | What it does | Output |
|------|-------------|--------|
| 3.1 | Feature tecniche (returns, volatility, RSI, MACD, BB) | `df_feat` |
| 3.2 | Feature di momentum cross-sectional (rank, z-score) | `df_feat` aggiornato |
| 3.3 | Feature fondamentali (P/E, EV/EBITDA, ROE, Debt/Eq) | `df_feat` aggiornato |
| 3.4 | Winsorize + missingness table + save | `Table_III_features.csv` |

**Intuizione economica**: Il momentum a 12-1 mesi cattura la persistenza dei rendimenti documentata da Jegadeesh & Titman (1993). I multipli fondamentali catturano premi di valore (Fama & French, 1992). La volatilità realizzata proxy-ata il rischio idiosincratico.

$$r_{t+1} = \\alpha + \\beta_1 \\text{Mom}_{12,1} + \\beta_2 \\text{Val} + \\beta_3 \\sigma_{21} + \\epsilon_{t+1}$$
"""))

new_cells.append(new_code_cell("""# ── 3.1  Feature Tecniche ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import mstats

logger.info("[3.1] Inizio feature engineering tecnico")

df = df_merged.copy().sort_values(["ticker", "date"])

def _winsorize(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1-p)
    return s.clip(lo, hi)

# ── Returns ──────────────────────────────────────────────────────────────────
for w in [1, 5, 21, 63, 126, 252]:
    col = f"ret_{w}d"
    df[col] = df.groupby("ticker")["adj_close"].pct_change(w).shift(1)

# ── Volatility realized ───────────────────────────────────────────────────────
for w in [21, 63]:
    daily_ret = df.groupby("ticker")["adj_close"].pct_change(1)
    df[f"vol_{w}d"] = daily_ret.groupby(df["ticker"]).transform(
        lambda x: x.rolling(w).std() * np.sqrt(252)
    ).shift(1)

# ── RSI (14) ──────────────────────────────────────────────────────────────────
def _rsi(series, n=14):
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.rolling(n).mean()
    avg_loss = loss.rolling(n).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

df["rsi_14"] = df.groupby("ticker")["adj_close"].transform(_rsi).shift(1)

# ── MACD ──────────────────────────────────────────────────────────────────────
def _macd_signal(series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return macd - signal  # histogram

df["macd_hist"] = df.groupby("ticker")["adj_close"].transform(_macd_signal).shift(1)

# ── Bollinger Band %B ─────────────────────────────────────────────────────────
def _bb_pct(series, n=20, k=2):
    ma  = series.rolling(n).mean()
    std = series.rolling(n).std()
    upper = ma + k * std
    lower = ma - k * std
    return (series - lower) / (upper - lower + 1e-9)

df["bb_pct"] = df.groupby("ticker")["adj_close"].transform(_bb_pct).shift(1)

# ── Price/SMA ratio ────────────────────────────────────────────────────────────
for w in [50, 200]:
    sma = df.groupby("ticker")["adj_close"].transform(lambda x: x.rolling(w).mean())
    df[f"price_sma{w}"] = (df["adj_close"] / sma).shift(1)

# ── Volume features ────────────────────────────────────────────────────────────
if "volume" in df.columns:
    vol_ma = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(21).mean())
    df["vol_ratio"] = (df["volume"] / vol_ma.replace(0, np.nan)).shift(1)

tech_cols = [c for c in df.columns if c not in ["date", "ticker", "adj_close",
             "open", "high", "low", "close", "volume", "adjusted"]]
logger.info(f"[3.1] Feature tecniche: {len(tech_cols)} colonne — esempio: {tech_cols[:6]}")
print(f"✅ [3.1] Feature tecniche: {len(tech_cols)} feature | shape: {df.shape}")
"""))

new_cells.append(new_code_cell("""# ── 3.2  Momentum Cross-Sectional (rank + z-score) ────────────────────────────
logger.info("[3.2] Momentum cross-sectional")

mom_cols = ["ret_21d", "ret_63d", "ret_126d", "ret_252d"]

for col in mom_cols:
    if col in df.columns:
        # Rank percentile cross-section per data
        df[f"{col}_xrank"] = df.groupby("date")[col].rank(pct=True)
        # Z-score cross-section
        df[f"{col}_xz"] = df.groupby("date")[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-9)
        )

# Composite momentum (Jegadeesh-Titman 12-1)
if "ret_252d" in df.columns and "ret_21d" in df.columns:
    df["mom_12_1"] = df["ret_252d"] - df["ret_21d"]
    df["mom_12_1_xrank"] = df.groupby("date")["mom_12_1"].rank(pct=True)

# Reversal short-term (1-week)
if "ret_5d" in df.columns:
    df["rev_5d_xrank"] = df.groupby("date")["ret_5d"].rank(pct=True)
    df["rev_5d_xrank"] = 1 - df["rev_5d_xrank"]  # reversal = short recent winners

print(f"✅ [3.2] Momentum cross-sectional: shape {df.shape}")
"""))

new_cells.append(new_code_cell("""# ── 3.3  Feature Fondamentali ─────────────────────────────────────────────────
logger.info("[3.3] Feature fondamentali")

feat_type = EXPERIMENT["ML_CONFIG"]["feature_set_type"]

if feat_type == "full" and "pe" not in df.columns:
    # Fondamentali non disponibili: costruiamo proxy sintetici con avviso
    logger.warning("[3.3] Fondamentali non disponibili: skip fundamental features (market_only mode)")
    print("⚠️  [3.3] Modalità market_only: feature fondamentali non disponibili")
    fund_cols = []
elif feat_type == "full":
    fund_map = {
        "pe_ratio"        : "pe",
        "ev_ebitda"       : "ev_ebitda",
        "pb_ratio"        : "pb",
        "roe"             : "roe",
        "roa"             : "roa",
        "debt_equity"     : "debt_eq",
        "current_ratio"   : "current_ratio",
        "gross_margin"    : "gross_margin",
        "net_margin"      : "net_margin",
        "revenue_growth"  : "rev_growth",
    }
    fund_cols = [v for v in fund_map.values() if v in df.columns]
    for c in fund_cols:
        df[f"{c}_xrank"] = df.groupby("date")[c].rank(pct=True)
    print(f"✅ [3.3] Feature fondamentali: {fund_cols}")
else:
    fund_cols = []
    print("ℹ️  [3.3] market_only mode: solo feature di prezzo")

print(f"   Shape totale df: {df.shape}")
"""))

new_cells.append(new_code_cell("""# ── 3.4  Winsorize + Missingness + Save df_feat ───────────────────────────────
logger.info("[3.4] Winsorize e salvataggio feature set")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)

# Colonne da escludere dal winsorize
exclude_win = {"date", "ticker", "adj_close", "open", "high", "low",
               "close", "volume", "adjusted", "calendarYear"}

num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in exclude_win]

# Winsorize 1-99 percentile
for c in num_cols:
    df[c] = _winsorize(df[c])

# Missingness table
miss = (df[num_cols].isna().sum() / len(df) * 100).sort_values(ascending=False)
miss_df = miss.reset_index()
miss_df.columns = ["feature", "pct_missing"]
miss_df.to_csv(OUTPUT_DIR / 'tables' / 'Table_III_missingness.csv', index=False)

print("Top 10 missing:")
print(miss_df.head(10).to_string(index=False))

# Save feature set
df_feat = df.copy()
df_feat.to_parquet(OUTPUT_DIR / 'tables' / 'df_feat.parquet', index=False)
logger.info(f"[3.4] df_feat salvato: shape={df_feat.shape}")
print(f"\n✅ [3.4] df_feat shape: {df_feat.shape}")
print(f"   Saved → {OUTPUT_DIR/'tables'/'df_feat.parquet'}")
"""))

print(f"Celle sezione 3 create: {len(new_cells)}")
print("Procedo con sezioni 4-14...")
"""))

nb.cells.extend(new_cells)

# Salva
with open(NB_PATH, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print(f"✅ Sezione 3 iniettata. Notebook ora ha {len(nb.cells)} celle.")
print("Ricarica il notebook per vedere le nuove celle (Runtime > Ricarica)")
"""}]

In [ ]:
# ============================================================
# 3.5 - Missingness table
# ============================================================

logger.info("[3.5] Missingness table")

feature_cols_final = [c for c in df_features.columns if c.endswith("_winsor")]

missingness = pd.DataFrame({
    "feature": feature_cols_final,
    "missing_count": [df_features[c].isna().sum() for c in feature_cols_final],
    "missing_pct": [df_features[c].isna().mean() for c in feature_cols_final],
})

missingness = missingness.sort_values("missing_pct", ascending=False).reset_index(drop=True)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
missingness_path = TABLES_DIR / "Table_3_feature_missingness.csv"
missingness.to_csv(missingness_path, index=False)

print("Saved:", missingness_path)
display(missingness.head(20))

In [135]:
# ============================================================
# 3.5 - Winsorize e salvataggio df_feat
# ============================================================

logger.info("[3.5] Winsorize e salvataggio df_feat")

from pathlib import Path

# usa i path già definiti in 0.4; se mancassero, ricostruiscili in modo robusto
if "OUTPUT_ROOT" not in globals():
    OUTPUT_ROOT = Path("/content/output")
if "TABLES_DIR" not in globals():
    TABLES_DIR = OUTPUT_ROOT / "tables"
if "FIGURES_DIR" not in globals():
    FIGURES_DIR = OUTPUT_ROOT / "figures"
if "LOGS_DIR" not in globals():
    LOGS_DIR = OUTPUT_ROOT / "logs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# alias retrocompatibile, così eventuale codice vecchio non si rompe
OUTPUT_DIR = OUTPUT_ROOT

2026-05-12 22:52:44,874 - INFO - [3.5] Winsorize e salvataggio df_feat
INFO:fundamental_valuation_dcf_definitive:[3.5] Winsorize e salvataggio df_feat


## 4. 🎯 Targets & Labels

> **Obiettivo**: definire le variabili dipendenti per i modelli ML e i modelli di valutazione.
> Ogni target è costruito con `.shift(-horizon)` sul rendimento futuro — mai sul passato.

| Cell | What it does | Output |
|------|-------------|--------|
| 4.1 | Return forward multi-orizzonte (1d, 5d, 21d, 63d) | `df_feat` aggiornato |
| 4.2 | Label binaria (up/down) e quintile cross-section | `df_feat` aggiornato |
| 4.3 | Target DCF: Free Cash Flow Yield, EV/EBITDA implied return | `df_feat` aggiornato |
| 4.4 | Save target columns + summary | `Table_IV_targets.csv` |

**Definizioni formali**

Return forward a orizzonte $h$:
$$y_{i,t}^{(h)} = \frac{P_{i,t+h} - P_{i,t}}{P_{i,t}}$$

Label binaria (classificazione):
$$L_{i,t} = \mathbf{1}\!\left[y_{i,t}^{(21)} > \text{median}_t\!\left(y^{(21)}\right)\right]$$

Quintile cross-sectional (ranking strategy):
$$Q_{i,t} = \text{quintile}\!\left(\text{rank}_{t}\!\left(y_{i,t}^{(21)}\right)\right) \in \{1,\ldots,5\}$$

In [138]:
# ── 4.1  Return forward multi-orizzonte ───────────────────────────────────────
import numpy as np
import pandas as pd

logger.info("[4.1] Target: return forward")

# Fallback: se df_feat non esiste (sezione 3 saltata) usiamo df_merged
if "df_feat" not in dir() or df_feat is None or df_feat.empty:
    logger.warning("[4.1] df_feat non trovato — uso df_merged come base")
    df_feat = df_merged.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
    print("⚠️  df_feat non trovato: uso df_merged come base")
else:
    df_feat = df_feat.sort_values(["ticker", "date"]).reset_index(drop=True)

# IMPORTANTE: shift NEGATIVO = futuro, nessun look-ahead leakage
for h in [1, 5, 21, 63]:
    col = f"fwd_ret_{h}d"
    df_feat[col] = (
        df_feat.groupby("ticker")["adj_close"]
        .pct_change(h)
        .shift(-h)
    )

logger.info(f"[4.1] OK — target columns: {[f'fwd_ret_{h}d' for h in [1,5,21,63]]}")
print(f"✅ [4.1] Return forward creati — shape: {df_feat.shape}")
print(f"   fwd_ret_21d: {df_feat['fwd_ret_21d'].describe().round(4).to_dict()}")

2026-05-12 22:56:53,416 - INFO - [4.1] Target: return forward
INFO:fundamental_valuation_dcf_definitive:[4.1] Target: return forward
2026-05-12 22:56:53,423 - WARNING - [4.1] df_feat non trovato — uso df_merged come base
/tmp/ipykernel_4687/3895335573.py:20: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(h)
/tmp/ipykernel_4687/3895335573.py:20: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(h)
/tmp/ipykernel_4687/3895335573.py:20: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future versio

⚠️  df_feat non trovato: uso df_merged come base
✅ [4.1] Return forward creati — shape: (140823, 16)
   fwd_ret_21d: {'count': 32939.0, 'mean': 0.0194, 'std': 0.1281, 'min': -0.583, '25%': -0.0196, '50%': 0.0095, '75%': 0.0418, 'max': 3.0477}


In [141]:
# ── 4.2  Label binaria (up/down) e quintile cross-sectional ───────────────────
import pandas as pd
import numpy as np
logger.info("[4.2] Target: label binaria e quintile")

# Fallback robusto per TARGET_HORIZON
TARGET_HORIZON = (
    EXPERIMENT.get("ML_CONFIG", {}).get("target_horizon_days")
    or EXPERIMENT.get("target_horizon_days")
    or 21
)
target_col = f"fwd_ret_{TARGET_HORIZON}d"
if target_col not in df_feat.columns:
    logger.warning(f"[4.2] {target_col} non trovato — uso fwd_ret_21d")
    target_col = "fwd_ret_21d"

# ── Label binaria: 1 se rendimento futuro > mediana cross-section ─────────────
df_feat["label_binary"] = (
    df_feat.groupby("date")[target_col]
    .transform(lambda x: (x > x.median()).astype(int))
)

# ── Quintile cross-sectional robusto (gestisce NaN e gruppi piccoli) ──────────
def _safe_quintile(x):
    """Assegna quintile solo se ci sono almeno 5 osservazioni non-NaN."""
    valid = x.dropna()
    if len(valid) < 5:
        return pd.Series(np.nan, index=x.index)
    try:
        result = pd.qcut(
            x.rank(method="first", na_option="keep"),
            5,
            labels=[1, 2, 3, 4, 5],
            duplicates="drop"
        ).astype(float)
        return result
    except Exception:
        return pd.Series(np.nan, index=x.index)

df_feat["label_quintile"] = (
    df_feat.groupby("date")[target_col]
    .transform(_safe_quintile)
    .astype(float)
)

# ── Excess return vs media universo ──────────────────────────────────────────
df_feat["fwd_excess_21d"] = df_feat.groupby("date")[target_col].transform(
    lambda x: x - x.mean()
)

pos_rate = df_feat["label_binary"].mean()
logger.info(f"[4.2] Label binaria — positive rate: {pos_rate:.2%}")
print(f"✅ [4.2] Label binaria e quintile creati")
print(f"   Positive rate (up): {pos_rate:.2%}  |  Target horizon: {TARGET_HORIZON}d")
print(f"   Quintile distribution:")
print(df_feat["label_quintile"].value_counts().sort_index().to_string())
print(f"   NaN quintile: {df_feat['label_quintile'].isna().sum()} righe")

2026-05-12 23:00:32,752 - INFO - [4.2] Target: label binaria e quintile
INFO:fundamental_valuation_dcf_definitive:[4.2] Target: label binaria e quintile
2026-05-12 23:00:47,413 - INFO - [4.2] Label binaria — positive rate: 10.74%
INFO:fundamental_valuation_dcf_definitive:[4.2] Label binaria — positive rate: 10.74%


✅ [4.2] Label binaria e quintile creati
   Positive rate (up): 10.74%  |  Target horizon: 21d
   Quintile distribution:
label_quintile
1.0    7571
2.0    5226
3.0    6289
4.0    5226
5.0    6667
   NaN quintile: 109844 righe


## 5. 📊 Descriptive Statistics

> **Obiettivo**: comprendere la distribuzione dei dati, identificare anomalie,
> e produrre le statistiche descrittive di base per il paper/report.

| Cell | What it does | Output |
|------|-------------|--------|
| 5.1 | Statistiche descrittive del panel | `Table_V_descriptive.csv` |
| 5.2 | Distribuzione rendimenti UCG vs universo | Figure `Fig_5.1_returns_dist.png` |
| 5.3 | Correlazione feature | Figure `Fig_5.2_corr_heatmap.png` |
| 5.4 | Timeline prezzi UCG + volumi | Figure `Fig_5.3_ucg_price.png` |

**Statistiche chiave**: media $\mu$, deviazione standard $\sigma$, skewness $\gamma_1$, kurtosis $\gamma_2$:

$$\gamma_1 = \frac{E[(X-\mu)^3]}{\sigma^3} \qquad \gamma_2 = \frac{E[(X-\mu)^4]}{\sigma^4} - 3$$

Un'elevata kurtosis positiva (leptocurtica) indica code spesse — tipica dei rendimenti azionari giornalieri.

## 6. 💰 Valuation Hub UniCredit (UCG.MI)

> **Obiettivo**: calcolare il *fair value* di UniCredit (UCG.MI) usando diversi modelli di valutazione, aggregare i risultati e visualizzare l'upside/downside.

| Cell | What it does | Output |
|------|-------------|--------|
| 6.1  | DCF multi-scenario (Bear/Base/Bull) | `df_dcf_valuation` |
| 6.2  | Dividend Discount Model (Gordon + H-Model) | `df_ddm_valuation` |
| 6.3  | Residual Income (RI) / Economic Value Added (EVA) | `df_ri_valuation` |
| 6.4  | Relative Valuation (P/E, P/B, EV/EBITDA vs peers) | `df_relative_valuation` |
| 6.5  | Price Target aggregato pesato + Upside/Downside | `Table_VI_price_target.csv` |
| 6.6  | Sensitivity Heatmap (WACC x g_terminal) | `Fig_VI_sensitivity_heatmap.png` |


### 6.1 DCF Multi-Scenario (Bear/Base/Bull)

Il Discounted Cash Flow (DCF) stima il valore intrinseco di un'azienda basandosi sui suoi futuri flussi di cassa liberi (FCF), attualizzati al costo medio ponderato del capitale (WACC).

$$ V_0 = \sum_{t=1}^{N} \frac{FCF_t}{(1 + WACC)^t} + \frac{TV_N}{(1 + WACC)^N} $$

Dove il Terminal Value (TV) è spesso calcolato con il modello di crescita perpetua di Gordon:

$$ TV_N = \frac{FCF_{N+1}}{(WACC - g)} $$


**Assunzioni chiave:**

*   **FCFF (Free Cash Flow to Firm):** I flussi di cassa disponibili per tutti i fornitori di capitale (azionisti e obbligazionisti).
*   **WACC (Weighted Average Cost of Capital):** Il costo medio che l'azienda sostiene per finanziarsi, ponderato per la quota di debito e capitale proprio.
*   **N:** Orizzonte di previsione esplicita (anni).
*   **g:** Tasso di crescita perpetua (Gordon Growth Rate).

Implementeremo tre scenari (Bear, Base, Bull) modificando le assunzioni chiave per la crescita dei ricavi, i margini operativi, la tassazione e il tasso di reinvestimento.

In [146]:
logger.info("[6.1] Inizio calcolo DCF multi-scenario per UCG.MI")

# --- 1. Estrazione dati UCG.MI -----------------------------------------------
# Robustly get UCG_TICKER, falling back to USER_SELECTION if EXPERIMENT is not fully populated
selected_company_raw = EXPERIMENT.get("selected_company", USER_SELECTION.get("selected_company", "UCG.MI"))
UCG_TICKER = selected_company_raw.replace('_', '.').upper()

df_ucg_prices = df_panel[df_panel["ticker"] == UCG_TICKER].copy()

if df_ucg_prices.empty:
    raise ValueError(f"Nessun dato di prezzo trovato per {UCG_TICKER}")

# Ultimo prezzo disponibile
latest_price = df_ucg_prices["adj_close"].iloc[-1]
logger.info(f"[6.1] Ultimo prezzo di {UCG_TICKER}: {latest_price:.2f}")

# Estrai info fondamentali per UCG.MI (dal context o default)
# Poiché `df_fund` è vuoto nel kernel state, useremo i valori da `EXPERIMENT` e il contesto utente.
ucg_fund_data = {
    "ticker": UCG_TICKER,
    "pe_ttm": EXPERIMENT.get("pe_ratio", 9.64), # Example, using provided context if available
    "pb": EXPERIMENT.get("pb_ratio", 1.53),
    "roe": EXPERIMENT.get("return_on_equity", 0.14),
    "beta": EXPERIMENT.get("beta", 1.092),
    "market_cap": EXPERIMENT.get("market_cap", 105e9), # 105B€
    "eps_ttm": EXPERIMENT.get("eps_ttm", 7.27),
    "eps_fwd": EXPERIMENT.get("eps_fwd", 8.19),
    "div_yield": EXPERIMENT.get("dividend_yield", 0.0445),
    "book_value_per_share": EXPERIMENT.get("book_value", 45.71),
    "shares_outstanding": EXPERIMENT.get("shares_outstanding", 105e9 / latest_price if latest_price else 1e9), # Estimate if not directly available
    "total_debt": EXPERIMENT.get("total_debt", 100e9), # Placeholder, assume value if not present in fundamentals
    "cash_and_equivalents": EXPERIMENT.get("cash_and_equivalents", 50e9), # Placeholder
    "ebitda": EXPERIMENT.get("ebitda", 15e9), # Placeholder
    "revenue": EXPERIMENT.get("revenue", 30e9), # Placeholder
    "operating_margin": EXPERIMENT.get("target_operating_margin", 0.17), # From cfa_balanced preset
    "tax_rate": EXPERIMENT.get("tax_rate", 0.25), # From EXPERIMENT
    "reinvestment_rate": EXPERIMENT.get("reinvestment_rate", 0.42) # From cfa_balanced preset
}

logger.info(f"[6.1] Dati fondamentali usati per {UCG_TICKER}: {json.dumps(ucg_fund_data, indent=2)}")

# --- 2. Parametri di valutazione ---------------------------------------------
forecast_horizon_years = EXPERIMENT["dcf_horizon_years"]
# Fix the KeyError by using .get() with a default value for risk_free_rate
risk_free_rate         = EXPERIMENT.get("risk_free_proxy_rate", EXPERIMENT.get("risk_free_rate", 0.03))
market_risk_premium    = EXPERIMENT["equity_risk_premium"]
beta                   = ucg_fund_data["beta"]
tax_rate               = ucg_fund_data["tax_rate"]

# --- 3. Scenari DCF ----------------------------------------------------------
scenarios = {
    "bear": {
        "sales_growth": [0.01, 0.005, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002], # Anni 1-10
        "operating_margin": [0.12, 0.11, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10], # Anni 1-10
        "reinvestment_rate": 0.30,
        "terminal_growth_rate": 0.005,
        "equity_risk_premium_adjust": 0.01 # +100bps per beta
    },
    "base": {
        "sales_growth": [0.03, 0.025, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02], # Anni 1-10
        "operating_margin": [0.17, 0.16, 0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.15], # Anni 1-10
        "reinvestment_rate": 0.40,
        "terminal_growth_rate": 0.015,
        "equity_risk_premium_adjust": 0.00
    },
    "bull": {
        "sales_growth": [0.05, 0.04, 0.035, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03], # Anni 1-10
        "operating_margin": [0.20, 0.19, 0.18, 0.18, 0.18, 0.18, 0.18, 0.18, 0.18, 0.18], # Anni 1-10
        "reinvestment_rate": 0.50,
        "terminal_growth_rate": 0.025,
        "equity_risk_premium_adjust": -0.005 # -50bps per beta
    }
}

# --- 4. Funzione di calcolo DCF ----------------------------------------------
def calculate_dcf(scenario_name, params):
    logger.info(f"[6.1] Calcolo DCF per scenario: {scenario_name}")

    # Cost of Equity (CAPM)
    adjusted_erp = market_risk_premium + params["equity_risk_premium_adjust"]
    cost_of_equity = risk_free_rate + beta * adjusted_erp

    # Cost of Debt (assunto per semplicità, in un vero modello si userebbe un rating o spread)
    cost_of_debt = EXPERIMENT.get("cost_of_debt", 0.045) # From cfa_balanced preset

    # Capital Structure (assunta se non in fondamentali, o da target)
    total_debt = ucg_fund_data["total_debt"]
    market_cap = ucg_fund_data["market_cap"]
    total_capital = market_cap + total_debt

    weight_equity = market_cap / total_capital
    weight_debt = total_debt / total_capital

    # WACC
    wacc = (weight_equity * cost_of_equity) + (weight_debt * cost_of_debt * (1 - tax_rate))

    logger.info(f"[6.1][{scenario_name}] CoE: {cost_of_equity:.4f}, CoD: {cost_of_debt:.4f}, WACC: {wacc:.4f}")

    # Proiezioni FCF
    projected_fcf = []
    current_revenue = ucg_fund_data["revenue"]
    current_ebitda = ucg_fund_data["ebitda"]

    # Assume FCF0 per il primo anno
    fcf_t0 = ucg_fund_data.get("free_cash_flow", current_ebitda * ucg_fund_data["operating_margin"] * (1 - tax_rate) * (1 - params["reinvestment_rate"])) # Crude FCF initial estimate

    for i in range(forecast_horizon_years):
        # Revenue Growth
        growth_rate = params["sales_growth"][i]
        current_revenue *= (1 + growth_rate)

        # Operating Income (EBIT) - simplified from operating margin
        ebit = current_revenue * params["operating_margin"][i]

        # Taxes
        taxes = ebit * tax_rate

        # NOPAT (Net Operating Profit After Tax)
        nopat = ebit - taxes

        # Reinvestment (simplified as a percentage of NOPAT or derived from growth)
        # A more complex model would use NOPAT / IC * (1 - reinvestment_rate)
        reinvestment = nopat * params["reinvestment_rate"]

        # Free Cash Flow to Firm
        fcf = nopat - reinvestment
        projected_fcf.append(fcf)

    logger.info(f"[6.1][{scenario_name}] FCF proiettati: {[f'{f:.2f}' for f in projected_fcf]}")

    # Terminal Value
    fcf_terminal_year = projected_fcf[-1]
    g_terminal = params["terminal_growth_rate"]

    if wacc <= g_terminal:
        logger.warning(f"[6.1][{scenario_name}] WACC ({wacc:.4f}) <= Tasso crescita perpetua ({g_terminal:.4f}). TV non calcolabile. Aumento WACC di 100bps per calcolo TV.")
        tv_wacc = wacc + 0.01 # Adjust WACC to allow calculation
    else:
        tv_wacc = wacc

    terminal_value = (fcf_terminal_year * (1 + g_terminal)) / (tv_wacc - g_terminal)
    logger.info(f"[6.1][{scenario_name}] Terminal Value: {terminal_value:.2f}")

    # Discount FCFs and Terminal Value
    pv_fcf = sum([fcf / ((1 + wacc)**(i + 1)) for i, fcf in enumerate(projected_fcf)])
    pv_terminal_value = terminal_value / ((1 + wacc)**forecast_horizon_years)

    enterprise_value = pv_fcf + pv_terminal_value

    # Equity Value
    equity_value = enterprise_value - ucg_fund_data["total_debt"] + ucg_fund_data["cash_and_equivalents"]

    fair_value_per_share = equity_value / ucg_fund_data["shares_outstanding"]

    return {
        "scenario": scenario_name,
        "fair_value": fair_value_per_share,
        "wacc": wacc,
        "terminal_growth": g_terminal,
        "latest_price": latest_price,
        "upside_pct": (fair_value_per_share - latest_price) / latest_price if latest_price else np.nan
    }

# --- 5. Esegui DCF per ogni scenario e aggrega risultati ---------------------
dcf_results = []
for scenario_name, params in scenarios.items():
    try:
        result = calculate_dcf(scenario_name, params)
        dcf_results.append(result)
    except Exception as e:
        logger.error(f"[6.1] Errore nel calcolo DCF per {scenario_name}: {e}")

df_dcf_valuation = pd.DataFrame(dcf_results)

# --- 6. Salva e mostra i risultati ------------------------------------------
TABLES_DIR.mkdir(parents=True, exist_ok=True)
dcf_output_path = TABLES_DIR / "Table_VI_1_dcf_valuation.csv"
dcf_output_path = TABLES_DIR / "Table_VI_1_dcf_valuation.csv"
df_dcf_valuation.to_csv(dcf_output_path, index=False)

logger.info(f"[6.1] Risultati DCF salvati in: {dcf_output_path}")
print(f"\n✅ [6.1] Calcolo DCF multi-scenario completato.")
print(f"   Risultati salvati in: {dcf_output_path}")
display(df_dcf_valuation)


2026-05-12 23:56:38,599 - INFO - [6.1] Inizio calcolo DCF multi-scenario per UCG.MI
INFO:fundamental_valuation_dcf_definitive:[6.1] Inizio calcolo DCF multi-scenario per UCG.MI
2026-05-12 23:56:38,632 - INFO - [6.1] Ultimo prezzo di UCG.MI: nan
INFO:fundamental_valuation_dcf_definitive:[6.1] Ultimo prezzo di UCG.MI: nan
2026-05-12 23:56:38,635 - INFO - [6.1] Dati fondamentali usati per UCG.MI: {
  "ticker": "UCG.MI",
  "pe_ttm": 9.64,
  "pb": 1.53,
  "roe": 0.14,
  "beta": 1.092,
  "market_cap": 105000000000.0,
  "eps_ttm": 7.27,
  "eps_fwd": 8.19,
  "div_yield": 0.0445,
  "book_value_per_share": 45.71,
  "shares_outstanding": NaN,
  "total_debt": 100000000000.0,
  "cash_and_equivalents": 50000000000.0,
  "ebitda": 15000000000.0,
  "revenue": 30000000000.0,
  "operating_margin": 0.17,
  "tax_rate": 0.25,
  "reinvestment_rate": 0.42
}
INFO:fundamental_valuation_dcf_definitive:[6.1] Dati fondamentali usati per UCG.MI: {
  "ticker": "UCG.MI",
  "pe_ttm": 9.64,
  "pb": 1.53,
  "roe": 0.14,


✅ [6.1] Calcolo DCF multi-scenario completato.
   Risultati salvati in: /content/output/tables/Table_VI_1_dcf_valuation.csv


,scenario,fair_value,wacc,terminal_growth,latest_price,upside_pct
0,bear,NaN,0.065388,0.005,NaN,NaN
1,base,NaN,0.059795,0.015,NaN,NaN
2,bull,NaN,0.056999,0.025,NaN,NaN


In [ ]:
# ============================================================
# 4 - Target engineering
# ============================================================

logger.info("4 Target engineering start")

if "dffeat" not in globals() or dffeat is None or dffeat.empty:
    raise ValueError("dffeat non disponibile: esegui prima la Section 3 canonica")

dffeat = dffeat.copy()
dffeat["date"] = pd.to_datetime(dffeat["date"], errors="coerce")
dffeat["ticker"] = dffeat["ticker"].astype(str).str.strip().str.upper()
dffeat = dffeat.sort_values(["ticker", "date"]).reset_index(drop=True)

price_col = "adjclose" if "adjclose" in dffeat.columns else "adjusted"
dffeat[price_col] = pd.to_numeric(dffeat[price_col], errors="coerce")

target_horizons = [1, 5, 21, 63, 126, 252]

for h in target_horizons:
    future_price = dffeat.groupby("ticker")[price_col].shift(-h)
    dffeat[f"fwdret{h}d"] = (future_price / dffeat[price_col]) - 1.0

TARGET_HORIZON = int(MLCONFIG.get("predictionhorizonmonths", 12) * 21)
if f"fwdret{TARGET_HORIZON}d" not in dffeat.columns:
    TARGET_HORIZON = 21

target_col = f"fwdret{TARGET_HORIZON}d"
EXPERIMENT["target_horizon_days"] = TARGET_HORIZON
EXPERIMENT["target_column"] = target_col

dffeat["target_regression"] = dffeat[target_col]

dffeat["label_binary"] = (
    dffeat.groupby("date")[target_col]
          .transform(lambda x: (x > x.median()).astype(float) if x.notna().sum() > 0 else np.nan)
)

def safe_quintile(x):
    valid = x.dropna()
    if len(valid) < 5:
        return pd.Series(np.nan, index=x.index)
    try:
        ranked = x.rank(method="first")
        out = pd.qcut(ranked, 5, labels=[1, 2, 3, 4, 5], duplicates="drop")
        return out.astype(float)
    except Exception:
        return pd.Series(np.nan, index=x.index)

dffeat["label_quintile"] = dffeat.groupby("date")[target_col].transform(safe_quintile)
dffeat["fwd_excess_21d"] = dffeat.groupby("date")["fwdret21d"].transform(lambda x: x - x.mean())

print("Target column:", target_col)
print("Target horizon days:", TARGET_HORIZON)
print("dffeat shape:", dffeat.shape)
display(dffeat[[c for c in ["date", "ticker", target_col, "target_regression", "label_binary", "label_quintile"] if c in dffeat.columns]].head(10))